# 🎓 Graduation Project — Phase 3
## A Comparative Study of Fraud Detection Models under Class Imbalance
### Dataset 3: Elliptic Bitcoin Transactions

---

**Author:** Nouha Laataoui | **Supervisor:** Nadjib Brahimi | **Institution:** Rennes School of Business

The Elliptic dataset introduces a structurally distinct fraud detection context compared to Phase 1 & 2:
- **Domain:** Cryptocurrency (Bitcoin blockchain) vs banking card transactions
- **Structure:** Graph-based transaction network — tabular features used for model consistency
- **Temporal:** 49 discrete time steps enabling unique temporal sensitivity analysis
- **Features:** 166 anonymized features (94 local + 72 aggregated graph neighbourhood features)
- **Labels:** illicit (fraud=1), licit (legit=0), unknown (~21% of nodes — excluded from supervised learning)
- **Fraud rate:** ~9.8% among labeled transactions (less extreme than Phase 1: 0.17%, more than Phase 2: 3.5%)

---

## ⚙️ 0 — Configuration & Setup

In [1]:
import sys
print(sys.executable)

import numpy as np
import shap
import lightgbm

print("numpy:", np.__version__)
print("shap:", shap.__version__)
print("lightgbm:", lightgbm.__version__)

c:\Users\Nouha Laataoui\anaconda3\envs\fraud-shap\python.exe
numpy: 1.26.4
shap: 0.46.0
lightgbm: 4.6.0


In [2]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# Graduation Project -- Phase 3 (Final)
# Dataset 3: Elliptic Bitcoin Transactions
# ============================================================

import os, time, json, warnings
from pathlib import Path
from collections import Counter
from itertools import combinations
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import ttest_rel, wilcoxon, ttest_ind
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_recall_curve, auc, average_precision_score,
    precision_recall_fscore_support, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

XGB_AVAILABLE = LGBM_AVAILABLE = False
try:
    import xgboost as xgb; XGB_AVAILABLE = True
except Exception: print("WARNING: xgboost not available.")
try:
    import lightgbm as lgb; LGBM_AVAILABLE = True
except Exception: print("WARNING: lightgbm not available.")

# ── CONFIG ───────────────────────────────────────────────────
SEED        = 42
TEST_SIZE   = 0.15
VAL_SIZE    = 0.15
N_REPEATS   = 5
N_BOOTSTRAP = 300
FRAUD_LABEL  = 1
NORMAL_LABEL = 0

IMB_RATIOS  = [0.001, 0.005, 0.01, 0.05]
SIZE_FRACS  = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
RECALL_GRID = np.linspace(0, 1, 500)

# ── Paths ────────────────────────────────────────────────────
BASE = Path(r"C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Datasets\Elliptic Bitcoin Transactions Dataset (Kaggle)")
FEATURES_CSV = BASE / "elliptic_txs_features.csv"
CLASSES_CSV  = BASE / "elliptic_txs_classes.csv"
EDGELIST_CSV = BASE / "elliptic_txs_edgelist.csv"

NOTEBOOK_DIR = Path(os.getcwd())
OUT_DIR = NOTEBOOK_DIR / "output phase 3"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# Phase 1 & 2 CSVs for 3-way cross-phase comparison
_p1_candidates = [
    NOTEBOOK_DIR / "output phase 1" / "results" / "phase1_baseline_results.csv",
    NOTEBOOK_DIR.parent / "output phase 1" / "results" / "phase1_baseline_results.csv",
    Path(r"C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\outputs phase 1\phase1\results\phase1_baseline_results.csv"),
]
_p2_candidates = [
    NOTEBOOK_DIR / "output phase 2" / "results" / "phase2_baseline_results.csv",
    NOTEBOOK_DIR.parent / "output phase 2" / "results" / "phase2_baseline_results.csv",
    Path(r"C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\outputs phase 2\phase2\results\phase2_baseline_results.csv"),
]
PHASE1_CSV = next((p for p in _p1_candidates if p.exists()), None)
PHASE2_CSV = next((p for p in _p2_candidates if p.exists()), None)

np.random.seed(SEED)
print(f"FEATURES_CSV : {FEATURES_CSV}")
print(f"OUTPUT       : {OUT_DIR}")
print(f"N_REPEATS    : {N_REPEATS}")
print(f"Phase 1 CSV  : {PHASE1_CSV or 'NOT FOUND — update _p1_candidates'}")
print(f"Phase 2 CSV  : {PHASE2_CSV or 'NOT FOUND — update _p2_candidates'}")

# ── Colorblind-safe palette (Wong 2011) — same as Phase 1 & 2 ───────────────
MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]
COLORS  = {"Logistic Regression": "#0072B2", "Random Forest": "#009E73",
           "XGBoost": "#E69F00",             "LightGBM": "#D55E00"}
MARKERS = {"Logistic Regression": "o", "Random Forest": "s",
           "XGBoost": "^",              "LightGBM": "D"}

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white",
    "axes.facecolor": "#F8F8F8", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.3, "grid.color": "#CCCCCC", "grid.linestyle": "--",
    "font.family": "DejaVu Sans", "font.size": 12,
    "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.labelsize": 12, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.fontsize": 10, "legend.framealpha": 0.9, "lines.linewidth": 2.2,
})

# ── Helpers (identical to Phase 1 & 2) ───────────────────────────────────────
from IPython.display import display

def save_fig(filename, tight=True):
    if tight: plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"  [saved] {filename}")

def styled_table(df, higher_is_better=None, title=""):
    if higher_is_better is None: higher_is_better = []
    def color_col(s, hib=True):
        vmin, vmax = s.min(), s.max()
        rng = vmax - vmin if vmax != vmin else 1
        colors = []
        for v in s:
            if not isinstance(v, (int, float)):
                colors.append(""); continue
            norm = (v - vmin) / rng
            if hib: r, g, b = int(255*(1-norm)), int(200*norm+55), 60
            else:   r, g, b = int(200*norm+55), int(255*(1-norm)), 60
            colors.append(f"background-color: rgba({r},{g},{b},0.25); font-weight: bold")
        return colors
    styler = df.style
    for col in df.columns:
        if col in higher_is_better:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=True), subset=[col])
        elif df[col].dtype in ["float64","float32"] and col not in ["threshold"]:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=False), subset=[col])
    styler = (styler
        .set_caption(f"<b>{title}</b>")
        .set_table_styles([
            {"selector":"caption","props":[("font-size","15px"),("text-align","left"),("padding","8px 0"),("color","#222")]},
            {"selector":"th","props":[("background","#2C3E50"),("color","white"),("padding","8px 12px"),("font-size","11px")]},
            {"selector":"td","props":[("padding","6px 12px"),("font-size","11px")]},
            {"selector":"tr:hover td","props":[("background-color","#EEF2FF")]},
        ]).format(precision=4))
    display(styler)

def print_summary(title, items):
    border = "─" * 60
    print(f"\n┌{border}┐")
    print(f"│  📊 {title:<55}│")
    print(f"├{border}┤")
    for k, v in items:
        line = f"  {k}: {v}"
        print(f"│{line:<60}│")
    print(f"└{border}┘")

def compute_pr_auc(y_true, y_proba):
    p, r, _ = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    return float(auc(r, p))

def find_threshold_f1(y_true, y_proba):
    p, r, thr = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    p, r = p[:-1], r[:-1]
    f1 = 2*(p*r)/(p+r+1e-12)
    return float(thr[np.argmax(f1)]) if len(thr) else 0.5

def evaluate(y_true, y_proba, threshold, model_name=""):
    y_pred = (y_proba >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=FRAUD_LABEL, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[NORMAL_LABEL, FRAUD_LABEL])
    tn, fp, fn, tp = cm.ravel()
    return dict(
        model=model_name, threshold=float(threshold),
        precision=float(p), recall=float(r), f1=float(f),
        pr_auc=compute_pr_auc(y_true, y_proba),
        ap=float(average_precision_score(y_true, y_proba)),
        roc_auc=float(roc_auc_score(y_true, y_proba)),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        specificity=float(tn/(tn+fp+1e-12)),
        fpr=float(fp/(fp+tn+1e-12))
    )

def bootstrap_metric(y_true, y_proba, threshold, metric_key="pr_auc",
                     n_boot=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(evaluate(y_true[idx], y_proba[idx], threshold)[metric_key])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def measure_infer_time(model, X_eval, repeats=3):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        model.predict_proba(X_eval)[:, 1]
        times.append(time.perf_counter() - t0)
    return float(np.mean(times))

def interp_pr(y_true, y_proba):
    p, r, _ = precision_recall_curve(y_true, y_proba, pos_label=FRAUD_LABEL)
    order = np.argsort(r)
    return np.interp(RECALL_GRID, r[order], p[order]), float(average_precision_score(y_true, y_proba))

def compute_ece(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_proba >= bins[i]) & (y_proba < bins[i+1])
        if mask.sum() == 0: continue
        ece += (mask.sum() / n) * abs(float(y_true[mask].mean()) - float(y_proba[mask].mean()))
    return ece

def build_model(name, seed, train_prev, feature_cols):
    prep = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler",  StandardScaler())])
    if name == "Logistic Regression":
        clf = LogisticRegression(solver="liblinear", class_weight="balanced",
                                 random_state=seed, max_iter=2000, C=0.1)
    elif name == "Random Forest":
        clf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                     random_state=seed, n_jobs=-1, max_depth=10,
                                     min_samples_split=50, min_samples_leaf=20,
                                     max_features="sqrt")
    elif name == "XGBoost":
        if not XGB_AVAILABLE: return None
        clf = xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.1,
                                 subsample=0.8, colsample_bytree=0.8,
                                 scale_pos_weight=(1-train_prev)/max(train_prev,1e-12),
                                 eval_metric="aucpr", random_state=seed,
                                 n_jobs=-1, verbosity=0)
    elif name == "LightGBM":
        if not LGBM_AVAILABLE: return None
        clf = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=31,
                                  subsample=0.8, colsample_bytree=0.8,
                                  class_weight="balanced", random_state=seed,
                                  n_jobs=-1, verbosity=-1)
    else:
        return None
    return Pipeline([("prep", prep), ("clf", clf)])


FEATURES_CSV : C:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Datasets\Elliptic Bitcoin Transactions Dataset (Kaggle)\elliptic_txs_features.csv
OUTPUT       : c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 3
N_REPEATS    : 5
Phase 1 CSV  : c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 1\results\phase1_baseline_results.csv
Phase 2 CSV  : c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 2\results\phase2_baseline_results.csv


## 📂 1 — Load Data

The Elliptic dataset consists of three files:
- **features:** 166 anonymized features + txId + time step for 203,769 transactions
- **classes:** labels — `illicit` (fraud), `licit` (legit), `unknown` (~21% — excluded from modeling)
- **edgelist:** ~234k directed edges between transactions (used for EDA only — tabular models don't use graph structure)

Unknown-labeled transactions are excluded from all supervised learning experiments to avoid label noise.

In [4]:
# ============================================================
# 1) LOAD DATA
# ============================================================
print("\n" + "="*70)
print("1) LOADING ELLIPTIC BITCOIN TRANSACTIONS")
print("="*70)

# Load features (no header row)
df_feat = pd.read_csv(FEATURES_CSV, header=None)
print(f"Features shape: {df_feat.shape}")

n_cols     = df_feat.shape[1]
n_features = n_cols - 2
n_local    = min(94, n_features)
n_agg      = n_features - n_local
print(f"Detected: {n_features} features ({n_local} local + {n_agg} aggregated)")

df_feat.columns = (["txId", "time_step"] +
                   [f"local_f{i}" for i in range(1, n_local + 1)] +
                   [f"agg_f{i}"   for i in range(1, n_agg + 1)])
df_feat["txId"] = df_feat["txId"].astype(str)

# Load class labels (has header: txId, class)
df_cls = pd.read_csv(CLASSES_CSV)
df_cls.columns = ["txId", "class"]
df_cls["txId"] = df_cls["txId"].astype(str)
print(f"Classes shape: {df_cls.shape}")
print(f"Class distribution (raw):\n{df_cls['class'].value_counts()}")

# Load edgelist for EDA
df_edges = pd.read_csv(EDGELIST_CSV, header=None, names=["txId1", "txId2"])
print(f"Edges: {len(df_edges):,}")

# Merge
df = df_feat.merge(df_cls, on="txId", how="inner")
print(f"\nMerged shape: {df.shape}")

# Map labels — strip whitespace, handle string values
# "1" = illicit (fraud=1), "2" = licit (legit=0), "unknown" = NaN (excluded)
df["class_clean"] = df["class"].astype(str).str.strip()
df["label"] = df["class_clean"].map({"1": 1, "2": 0, "unknown": np.nan})

# Labeled only
df_labeled = df[df["label"].notna()].copy()
df_labeled["label"] = df_labeled["label"].astype(int)
df_labeled["txId"] = df_labeled["txId"].astype(str)  # keep consistent with edgelist

print(f"Labeled transactions: {len(df_labeled):,}")
print(f"  Illicit (fraud): {(df_labeled['label']==1).sum():,}")
print(f"  Licit (legit):   {(df_labeled['label']==0).sum():,}")
print(f"  Fraud rate:      {df_labeled['label'].mean():.4%}")
print(f"  Unknown excluded: {df['label'].isna().sum():,} ({df['label'].isna().mean():.1%})")

FEATURE_COLS = [c for c in df_labeled.columns
                if c.startswith("local_f") or c.startswith("agg_f")]
print(f"\nFeature columns: {len(FEATURE_COLS)}")
print(f"Time steps: {df_labeled['time_step'].nunique()} "
      f"(steps {int(df_labeled['time_step'].min())}–{int(df_labeled['time_step'].max())})")

X = df_labeled[FEATURE_COLS].copy()
y = df_labeled["label"].copy()

print_summary("Data Loading", [
    ("Total transactions", f"{len(df):,}"),
    ("Labeled transactions", f"{len(df_labeled):,}"),
    ("Fraud (illicit)", f"{y.sum():,} ({y.mean():.4%})"),
    ("Imbalance ratio", f"{(y==0).sum()//(y==1).sum()}:1"),
    ("Time steps", f"{df_labeled['time_step'].nunique()}"),
    ("Features", f"{len(FEATURE_COLS)} ({n_local} local + {n_agg} aggregated)"),
    ("Edges in graph", f"{len(df_edges):,}"),
])



1) LOADING ELLIPTIC BITCOIN TRANSACTIONS
Features shape: (203769, 167)
Detected: 165 features (94 local + 71 aggregated)
Classes shape: (203769, 2)
Class distribution (raw):
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64
Edges: 234,356

Merged shape: (203769, 168)
Labeled transactions: 46,564
  Illicit (fraud): 4,545
  Licit (legit):   42,019
  Fraud rate:      9.7608%
  Unknown excluded: 157,205 (77.1%)

Feature columns: 165
Time steps: 49 (steps 1–49)

┌────────────────────────────────────────────────────────────┐
│  📊 Data Loading                                           │
├────────────────────────────────────────────────────────────┤
│  Total transactions: 203,769                               │
│  Labeled transactions: 46,564                              │
│  Fraud (illicit): 4,545 (9.7608%)                          │
│  Imbalance ratio: 9:1                                      │
│  Time steps: 49                                            

## 🔍 2 — Exploratory Data Analysis

Key questions specific to Elliptic:
- How does the fraud rate vary across time steps? (temporal pattern unique to this dataset)
- How do features distinguish illicit from licit transactions?
- How does the fraud prevalence compare to Phase 1 (0.17%) and Phase 2 (3.5%)?
- What is the graph connectivity structure?

In [5]:
# ============================================================
# 2) EDA
# ============================================================
print("\n" + "="*70)
print("2) EDA — ELLIPTIC BITCOIN")
print("="*70)

# ── 2a. Class distribution + cross-phase comparison ─────────
counts = y.value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Class Distribution — Elliptic Bitcoin vs Phase 1 & 2",
             fontsize=14, fontweight="bold")

axes[0].bar(["Licit (0)", "Illicit (1)"],
            [counts.get(0,0), counts.get(1,0)],
            color=["#0072B2", "#D55E00"], edgecolor="black")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Elliptic  |  Fraud rate = {y.mean():.2%}")
for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
    axes[0].text(i, v*1.01, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie([counts.get(0,0), counts.get(1,0)],
            labels=["Licit", "Illicit"],
            colors=["#0072B2", "#D55E00"],
            autopct=lambda p: f"{p:.2f}%",
            explode=[0, 0.08], startangle=90)
axes[1].set_title("Proportions")

datasets = ["Kaggle CC\n(Phase 1, 0.17%)", "IEEE-CIS\n(Phase 2, 3.5%)",
            "Elliptic\n(Phase 3)"]
rates = [0.1727, 3.5, y.mean()*100]
bars = axes[2].bar(datasets, rates,
                   color=["#0072B2", "#009E73", "#D55E00"],
                   edgecolor="black", alpha=0.85)
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].set_title("Fraud Prevalence: All 3 Phases")
for bar, val in zip(bars, rates):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                 f"{val:.4f}%", ha="center", fontweight="bold", fontsize=9)
save_fig("eda_01_class_distribution.png")

# ── 2b. Temporal analysis — unique to Elliptic ──────────────        
time_stats = df.groupby("time_step").agg(
    total=("label", "count"),
    labeled=("label", lambda x: x.notna().sum()),
    illicit=("label", lambda x: (x==1).sum()),
    licit=("label", lambda x: (x==0).sum()),
    unknown=("label", lambda x: x.isna().sum()),
).reset_index()
time_stats["fraud_rate"] = time_stats["illicit"] / time_stats["labeled"].replace(0, np.nan) * 100

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
fig.suptitle("Temporal Analysis — Elliptic Bitcoin (49 Time Steps)",
             fontsize=14, fontweight="bold")

# Transaction volume over time
axes[0,0].stackplot(time_stats["time_step"],
                    time_stats["licit"], time_stats["illicit"], time_stats["unknown"],
                    labels=["Licit", "Illicit", "Unknown"],
                    colors=["#0072B2", "#D55E00", "#888888"], alpha=0.8)
axes[0,0].set_xlabel("Time Step"); axes[0,0].set_ylabel("Transaction Count")
axes[0,0].set_title("Transaction Volume by Label Over Time")
axes[0,0].legend(fontsize=9)

# Fraud rate over time
axes[0,1].plot(time_stats["time_step"], time_stats["fraud_rate"],
               color="#D55E00", linewidth=2.5, marker="o", markersize=4)
axes[0,1].axhline(y.mean()*100, color="gray", linestyle="--",
                  label=f"Overall mean ({y.mean()*100:.1f}%)")
axes[0,1].set_xlabel("Time Step"); axes[0,1].set_ylabel("Fraud Rate (%)")
axes[0,1].set_title("Fraud Rate (%) per Time Step")
axes[0,1].legend()

# Highlight time steps with unusually high fraud
peak_steps = time_stats.nlargest(5, "fraud_rate")
for _, row in peak_steps.iterrows():
    axes[0,1].annotate(f"t={int(row['time_step'])}\n{row['fraud_rate']:.0f}%",
                       xy=(row["time_step"], row["fraud_rate"]),
                       xytext=(row["time_step"]+1, row["fraud_rate"]+2),
                       fontsize=7, color="#D55E00")

# Unknown ratio over time
axes[1,0].bar(time_stats["time_step"],
              time_stats["unknown"]/time_stats["total"]*100,
              color="#888888", edgecolor="black", alpha=0.7)
axes[1,0].set_xlabel("Time Step"); axes[1,0].set_ylabel("Unknown %")
axes[1,0].set_title("Unknown (Unlabeled) Transaction % per Time Step")

# Labeled vs total
axes[1,1].plot(time_stats["time_step"], time_stats["total"], label="Total",
               color="#0072B2", linewidth=2)
axes[1,1].plot(time_stats["time_step"], time_stats["labeled"], label="Labeled",
               color="#009E73", linewidth=2, linestyle="--")
axes[1,1].set_xlabel("Time Step"); axes[1,1].set_ylabel("Count")
axes[1,1].set_title("Total vs Labeled Transactions per Time Step")
axes[1,1].legend()
save_fig("eda_02_temporal_analysis.png")

# ── 2c. Feature distributions — local vs aggregated ─────────
local_feats = [c for c in FEATURE_COLS if c.startswith("local_")]
agg_feats   = [c for c in FEATURE_COLS if c.startswith("agg_")]

print(f"  Local features: {len(local_feats)}")
print(f"  Aggregated features: {len(agg_feats)}")

# Missing values check
missing = df_labeled[FEATURE_COLS].isnull().sum()
print(f"  Missing values: {missing.sum()} (all features complete: {missing.sum()==0})")

# t-test on labeled data to find discriminative features
from scipy.stats import ttest_ind as ttest_ind_feat
rows = []
for col in FEATURE_COLS:
    x0 = df_labeled.loc[df_labeled["label"]==0, col].values
    x1 = df_labeled.loc[df_labeled["label"]==1, col].values
    try:
        _, p_val = ttest_ind_feat(x0, x1, equal_var=False, nan_policy="omit")
        d = float((np.mean(x1)-np.mean(x0)) / (np.sqrt((np.std(x0)**2+np.std(x1)**2)/2)+1e-12))
        rows.append({"feature": col, "p_value": p_val, "cohen_d": d})
    except Exception:
        pass
feat_stats = pd.DataFrame(rows).sort_values("p_value")
feat_stats.to_csv(RES_DIR / "eda_feature_ttests.csv", index=False)

# Top 6 discriminative features violin plots
top6 = feat_stats.head(6)["feature"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle("Top 6 Most Discriminative Features (by t-test p-value) — Elliptic",
             fontsize=14, fontweight="bold")
for ax, feat in zip(axes.flatten(), top6):
    d_val = feat_stats.loc[feat_stats["feature"]==feat, "cohen_d"].values[0]
    p_val = feat_stats.loc[feat_stats["feature"]==feat, "p_value"].values[0]
    plot_df = pd.DataFrame({
        "value": df_labeled[feat],
        "Class": df_labeled["label"].map({0:"Licit", 1:"Illicit"})
    })
    sns.violinplot(data=plot_df, x="Class", y="value",
                   palette={"Licit":"#0072B2","Illicit":"#D55E00"},
                   ax=ax, inner="quartile")
    ax.set_title(f"{feat}  |  d={d_val:.2f}  p={p_val:.2e}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("")
save_fig("eda_03_top6_feature_violins.png")

# ── 2d. Graph structure overview ────────────────────────────
# Degree distribution of labeled nodes
if EDGELIST_CSV.exists():
    edge_nodes = pd.concat([df_edges["txId1"], df_edges["txId2"]])
    degree = edge_nodes.value_counts().reset_index()
    degree.columns = ["txId", "degree"]
    degree_labeled = degree.merge(df_labeled[["txId","label"]], on="txId", how="inner")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Graph Structure — Elliptic Bitcoin Network",
                 fontsize=14, fontweight="bold")

    for cls, color, label in [(0,"#0072B2","Licit"),(1,"#D55E00","Illicit")]:
        sub = degree_labeled[degree_labeled["label"]==cls]["degree"]
        axes[0].hist(np.log1p(sub), bins=40, alpha=0.6,
                     color=color, label=label, density=True)
    axes[0].set_xlabel("log(1 + degree)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Transaction Degree Distribution by Class")
    axes[0].legend()

    # Average degree per time step
    deg_per_ts = degree_labeled.merge(
        df_labeled[["txId","time_step"]], on="txId").groupby(
        ["time_step","label"])["degree"].mean().reset_index()
    for cls, color, lbl in [(0,"#0072B2","Licit"),(1,"#D55E00","Illicit")]:
        sub = deg_per_ts[deg_per_ts["label"]==cls]
        axes[1].plot(sub["time_step"], sub["degree"],
                     color=color, linewidth=2, label=lbl)
    axes[1].set_xlabel("Time Step")
    axes[1].set_ylabel("Mean Degree")
    axes[1].set_title("Mean Node Degree per Time Step")
    axes[1].legend()
    save_fig("eda_04_graph_structure.png")

print_summary("EDA Summary", [
    ("Local features", f"{len(local_feats)}"),
    ("Aggregated features", f"{len(agg_feats)}"),
    ("Missing values", "0 (complete dataset)"),
    ("Top discriminative feature", feat_stats.iloc[0]["feature"]),
    ("Time steps", f"{df_labeled['time_step'].nunique()}"),
    ("Peak fraud time step", f"t={int(time_stats.loc[time_stats['fraud_rate'].idxmax(),'time_step'])} ({time_stats['fraud_rate'].max():.1f}%)"),
    ("Graph edges", f"{len(df_edges):,}"),
])



2) EDA — ELLIPTIC BITCOIN
  [saved] eda_01_class_distribution.png
  [saved] eda_02_temporal_analysis.png
  Local features: 94
  Aggregated features: 71
  Missing values: 0 (all features complete: True)
  [saved] eda_03_top6_feature_violins.png
  [saved] eda_04_graph_structure.png

┌────────────────────────────────────────────────────────────┐
│  📊 EDA Summary                                            │
├────────────────────────────────────────────────────────────┤
│  Local features: 94                                        │
│  Aggregated features: 71                                   │
│  Missing values: 0 (complete dataset)                      │
│  Top discriminative feature: local_f83                     │
│  Time steps: 49                                            │
│  Peak fraud time step: t=13 (36.0%)                        │
│  Graph edges: 234,356                                      │
└────────────────────────────────────────────────────────────┘


## ✂️ 3 — Stratified Train / Validation / Test Split

Same 70/15/15 stratified split as Phase 1 & 2 for cross-phase comparability.

> **Note on temporal leakage:** For this baseline analysis, we use a random stratified split (consistent with Phase 1 & 2) rather than a time-based split. This is a deliberate methodological choice for comparability. A temporal split analysis is added in Section 7 (Sensitivity) to assess the impact of this decision.

In [6]:
# ============================================================
# 3) DATA SPLIT (70 / 15 / 15)
# ============================================================
print("\n" + "="*70)
print("3) DATA SPLIT (70 / 15 / 15)")
print("="*70)

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
val_frac = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=val_frac, stratify=y_tmp, random_state=SEED)

train_prev = float(y_train.mean())

print(f"Train : {len(X_train):,}  ({y_train.mean():.4%} fraud)")
print(f"Val   : {len(X_val):,}  ({y_val.mean():.4%} fraud)")
print(f"Test  : {len(X_test):,}  ({y_test.mean():.4%} fraud)")

print_summary("Data Split", [
    ("Train set", f"{len(X_train):,} rows  |  {y_train.mean():.4%} fraud"),
    ("Validation set", f"{len(X_val):,} rows  |  {y_val.mean():.4%} fraud"),
    ("Test set", f"{len(X_test):,} rows  |  {y_test.mean():.4%} fraud"),
    ("Imbalance strategy", "class_weight=balanced (no SMOTE/resampling)"),
    ("Split method", "Stratified random (consistent with Phase 1 & 2)"),
])



3) DATA SPLIT (70 / 15 / 15)
Train : 32,594  (9.7595% fraud)
Val   : 6,985  (9.7638% fraud)
Test  : 6,985  (9.7638% fraud)

┌────────────────────────────────────────────────────────────┐
│  📊 Data Split                                             │
├────────────────────────────────────────────────────────────┤
│  Train set: 32,594 rows  |  9.7595% fraud                  │
│  Validation set: 6,985 rows  |  9.7638% fraud              │
│  Test set: 6,985 rows  |  9.7638% fraud                    │
│  Imbalance strategy: class_weight=balanced (no SMOTE/resampling)│
│  Split method: Stratified random (consistent with Phase 1 & 2)│
└────────────────────────────────────────────────────────────┘


## 🏆 4 — Baseline Benchmark

Same four models as Phase 1 & 2, with identical hyperparameters and evaluation protocol.
Primary metric: PR-AUC. Threshold: max-F1 on validation. Uncertainty: 95% bootstrap CI.

In [7]:
# ============================================================
# 4) BASELINE BENCHMARK
# ============================================================
print("\n" + "="*70)
print("4) BASELINE BENCHMARK")
print("="*70)

baseline_rows = []
pr_cache    = {}
thr_cache   = {}
model_cache = {}

for mname in MODEL_NAMES:
    model = build_model(mname, SEED, train_prev, FEATURE_COLS)
    if model is None: continue
    print(f"  Training {mname}...", end=" ", flush=True)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_t = time.perf_counter() - t0

    val_proba  = model.predict_proba(X_val)[:, 1]
    thr        = find_threshold_f1(y_val.values, val_proba)
    test_proba = model.predict_proba(X_test)[:, 1]
    infer_t    = measure_infer_time(model, X_test)

    met = evaluate(y_test.values, test_proba, thr, model_name=mname)
    met["train_time_s"]     = float(train_t)
    met["infer_time_s"] = float(infer_t)

    ci_pr_lo, ci_pr_hi = bootstrap_metric(y_test.values, test_proba, thr, "pr_auc")
    ci_ap_lo, ci_ap_hi = bootstrap_metric(y_test.values, test_proba, thr, "ap")
    ci_f1_lo, ci_f1_hi = bootstrap_metric(y_test.values, test_proba, thr, "f1")
    met.update({"pr_auc_ci_lo": ci_pr_lo, "pr_auc_ci_hi": ci_pr_hi,
                "ap_ci_lo": ci_ap_lo, "ap_ci_hi": ci_ap_hi,
                "f1_ci_lo": ci_f1_lo, "f1_ci_hi": ci_f1_hi})

    baseline_rows.append(met)
    pr_cache[mname]    = test_proba
    thr_cache[mname]   = thr
    model_cache[mname] = model
    print(f"done ({train_t:.1f}s) | PR-AUC={met['pr_auc']:.4f} | F1={met['f1']:.4f}")

baseline_df = pd.DataFrame(baseline_rows).sort_values("pr_auc", ascending=False)
baseline_df.to_csv(RES_DIR / "phase3_baseline_results.csv", index=False)

# ── Figures ──────────────────────────────────────────────────
# PR Curves
fig, ax = plt.subplots(figsize=(9, 7))
chance = float(y_test.mean())
ax.hlines(chance, 0, 1, linestyles="--", colors="gray", linewidth=1.5,
          label=f"No Skill (AP={chance:.4f})")
for mname, proba in pr_cache.items():
    p, r, _ = precision_recall_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    ap = average_precision_score(y_test.values, proba)
    ax.plot(r, p, color=COLORS[mname], linewidth=2, label=f"{mname}  AP={ap:.4f}")
    thr = thr_cache[mname]
    y_pred_thr = (proba >= thr).astype(int)
    rec_op = float((y_pred_thr[y_test.values==1]).sum()) / max(y_test.sum(), 1)
    prec_op = float(y_pred_thr[y_test.values==1].mean()) if y_pred_thr[y_test.values==1].sum() else 0
    ax.scatter([rec_op], [prec_op], marker=MARKERS[mname], s=90,
               color=COLORS[mname], zorder=5)
ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision–Recall Curves (Test Set)\n★ = Operating threshold (max-F1 on val)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1)
save_fig("baseline_01_pr_curves.png")

# ROC Curves
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0,1],[0,1],"k--", linewidth=1.2, label="Random Classifier")
for mname, proba in pr_cache.items():
    fpr_r, tpr_r, _ = roc_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    roc = roc_auc_score(y_test.values, proba)
    ax.plot(fpr_r, tpr_r, color=COLORS[mname], linewidth=2,
            label=f"{mname}  ROC-AUC={roc:.4f}")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (Test Set)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1)
save_fig("baseline_02_roc_curves.png")

# Confusion matrices
n_avail = len(pr_cache)
fig, axes = plt.subplots(1, n_avail, figsize=(5*n_avail, 5))
fig.suptitle("Confusion Matrices (Test Set)", fontsize=13, fontweight="bold")
if n_avail == 1: axes = [axes]
for ax, mname in zip(axes, pr_cache):
    proba = pr_cache[mname]; thr = thr_cache[mname]
    y_pred = (proba >= thr).astype(int)
    cm_plot = confusion_matrix(y_test.values, y_pred, labels=[NORMAL_LABEL, FRAUD_LABEL])
    cm_pct  = cm_plot.astype(float) / cm_plot.sum(axis=1, keepdims=True) * 100
    annot = np.array([[f"{v}\n({p:.1f}%)" for v,p in zip(row_c,row_p)]
                      for row_c,row_p in zip(cm_plot, cm_pct)])
    sns.heatmap(cm_plot, annot=annot, fmt="", cmap="Blues",
                xticklabels=["Licit","Illicit"],
                yticklabels=["Licit","Illicit"],
                cbar=False, linewidths=0.5, ax=ax, annot_kws={"size":11})
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"{mname}\nF1={baseline_df.loc[baseline_df['model']==mname,'f1'].values[0]:.4f}")
save_fig("baseline_03_confusion_matrices.png")

# Performance bars with CI
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Model Performance with 95% Bootstrap CI (Test Set)",
             fontsize=13, fontweight="bold")
for ax, (metric, ylabel, ci_lo_col, ci_hi_col) in zip(axes, [
    ("pr_auc","PR-AUC","pr_auc_ci_lo","pr_auc_ci_hi"),
    ("ap","Average Precision","ap_ci_lo","ap_ci_hi"),
    ("f1","F1-Score","f1_ci_lo","f1_ci_hi"),
]):
    sub = baseline_df.set_index("model").reindex(
        [m for m in MODEL_NAMES if m in baseline_df["model"].values]
    ).dropna(subset=[metric])
    means  = sub[metric].values
    errors = np.array([means - sub[ci_lo_col].values, sub[ci_hi_col].values - means])
    names  = sub.index.tolist()
    bars = ax.bar(names, means, yerr=errors,
                  color=[COLORS.get(m,"steelblue") for m in names],
                  edgecolor="black", alpha=0.85,
                  capsize=5, error_kw={"ecolor":"black","linewidth":1.5})
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"{ylabel} Comparison", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1); ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")
save_fig("baseline_04_performance_bars_CI.png")

# Timing
fig, ax = plt.subplots(figsize=(10, 5))
sub_op = baseline_df.set_index("model").reindex(
    [m for m in MODEL_NAMES if m in baseline_df["model"].values])
x = np.arange(len(sub_op)); w = 0.35
b1 = ax.bar(x-w/2, sub_op["train_time_s"],    width=w, label="Train Time (s)",
            color="#0072B2", edgecolor="black", alpha=0.85)
b2 = ax.bar(x+w/2, sub_op["infer_time_s"], width=w, label="Inference Time (s)",
            color="#E69F00", edgecolor="black", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(sub_op.index, rotation=15)
ax.set_ylabel("Time (seconds)")
ax.set_title("Computational Cost Comparison", fontsize=12, fontweight="bold")
ax.legend()
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h*1.02, f"{h:.2f}s",
            ha="center", fontsize=9)
save_fig("baseline_05_timing.png")

# Styled results table
display_df = baseline_df[["model","pr_auc","pr_auc_ci_lo","pr_auc_ci_hi",
                           "ap","f1","f1_ci_lo","f1_ci_hi",
                           "recall","precision","roc_auc","train_time_s"]].copy()
display_df.columns = ["Model","PR-AUC","PR-AUC CI↓","PR-AUC CI↑",
                       "AP","F1","F1 CI↓","F1 CI↑",
                       "Recall","Precision","ROC-AUC","Train(s)"]
styled_table(display_df,
             higher_is_better=["PR-AUC","AP","F1","Recall","Precision","ROC-AUC"],
             title="Baseline Results — Test Set (Elliptic Bitcoin)")

# -- Summary — dynamic, works for any set of models --------------------------
best = baseline_df.iloc[0]

summary_rows = [("Best model (PR-AUC)", f"{best['model']}  →  PR-AUC={best['pr_auc']:.4f}")]

for mname in baseline_df["model"].values:
    pr = baseline_df[baseline_df["model"] == mname]["pr_auc"].values[0]
    summary_rows.append((f"{mname} PR-AUC", f"{pr:.4f}"))

summary_rows.append(
    ("Fastest model", f"{baseline_df.loc[baseline_df['train_time_s'].idxmin(), 'model']} "
                      f"({baseline_df['train_time_s'].min():.2f}s)")
)

print_summary("Baseline Benchmark — Elliptic Bitcoin", summary_rows)


4) BASELINE BENCHMARK
  Training Logistic Regression... done (5.8s) | PR-AUC=0.6721 | F1=0.7160
  Training Random Forest... done (2.2s) | PR-AUC=0.9679 | F1=0.9251
  Training XGBoost... done (6.0s) | PR-AUC=0.9812 | F1=0.9372
  Training LightGBM... done (2.2s) | PR-AUC=0.9848 | F1=0.9507
  [saved] baseline_01_pr_curves.png
  [saved] baseline_02_roc_curves.png
  [saved] baseline_03_confusion_matrices.png
  [saved] baseline_04_performance_bars_CI.png
  [saved] baseline_05_timing.png


,Model,PR-AUC,PR-AUC CI↓,PR-AUC CI↑,AP,F1,F1 CI↓,F1 CI↑,Recall,Precision,ROC-AUC,Train(s)
3,LightGBM,0.9848,0.9788,0.9895,0.9848,0.9507,0.9395,0.9611,0.9326,0.9695,0.9975,2.2499
2,XGBoost,0.9812,0.9745,0.9869,0.9812,0.9372,0.9223,0.9516,0.9076,0.9687,0.9968,5.9620
1,Random Forest,0.9679,0.9588,0.9761,0.9679,0.9251,0.9115,0.9390,0.8783,0.9772,0.9935,2.1886
0,Logistic Regression,0.6721,0.6333,0.7150,0.6729,0.7160,0.6888,0.7460,0.6950,0.7383,0.9520,5.8056



┌────────────────────────────────────────────────────────────┐
│  📊 Baseline Benchmark — Elliptic Bitcoin                  │
├────────────────────────────────────────────────────────────┤
│  Best model (PR-AUC): LightGBM  →  PR-AUC=0.9848           │
│  LightGBM PR-AUC: 0.9848                                   │
│  XGBoost PR-AUC: 0.9812                                    │
│  Random Forest PR-AUC: 0.9679                              │
│  Logistic Regression PR-AUC: 0.6721                        │
│  Fastest model: Random Forest (2.19s)                      │
└────────────────────────────────────────────────────────────┘


## 🔁 5 — Robustness Analysis

Models retrained across N=5 random seeds with different train/val/test splits.
Mean ± std estimates over repeated experiments.

In [8]:
# ============================================================
# 5) ROBUSTNESS — REPEATED EXPERIMENTS
# ============================================================
print("\n" + "="*70)
print(f"5) ROBUSTNESS — REPEATED EXPERIMENTS (N={N_REPEATS})")
print("="*70)

pr_curves_data = {m: {"precisions": [], "aps": []} for m in MODEL_NAMES}
repeat_rows    = []

for rep in range(N_REPEATS):
    seed = SEED + rep
    print(f"  Repeat {rep+1}/{N_REPEATS} (seed={seed})...", flush=True)
    X_tmp_r, X_test_r, y_tmp_r, y_test_r = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    vf = VAL_SIZE / (1.0 - TEST_SIZE)
    X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
        X_tmp_r, y_tmp_r, test_size=vf, stratify=y_tmp_r, random_state=seed)
    tp = float(y_train_r.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, seed, tp, FEATURE_COLS)
        if model is None: continue
        t0 = time.perf_counter()
        model.fit(X_train_r, y_train_r)
        tr_t = time.perf_counter() - t0
        val_p   = model.predict_proba(X_val_r)[:, 1]
        thr     = find_threshold_f1(y_val_r.values, val_p)
        test_p  = model.predict_proba(X_test_r)[:, 1]
        infer_t = measure_infer_time(model, X_test_r)

        met = evaluate(y_test_r.values, test_p, thr, model_name=mname)
        met.update({"repeat": rep, "seed": seed,
                    "train_time_s": tr_t, "infer_time_s": infer_t})
        repeat_rows.append(met)

        ip, ap_r = interp_pr(y_test_r.values, test_p)
        pr_curves_data[mname]["precisions"].append(ip)
        pr_curves_data[mname]["aps"].append(ap_r)

repeats_df = pd.DataFrame(repeat_rows)
repeats_df.to_csv(RES_DIR / "phase3_repeats_all_runs.csv", index=False)

# Mean PR curves
fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(float(y.mean()), 0, 1, linestyles="--", colors="gray",
          linewidth=1.5, label=f"No Skill (AP={y.mean():.4f})")
for mname in MODEL_NAMES:
    data = pr_curves_data[mname]
    if not data["precisions"]: continue
    pm = np.array(data["precisions"])
    mean_p = pm.mean(axis=0); std_p = pm.std(axis=0)
    mean_ap = np.mean(data["aps"]); std_ap = np.std(data["aps"])
    ax.plot(RECALL_GRID, mean_p, color=COLORS[mname], linewidth=2,
            label=f"{mname}  AP={mean_ap:.4f}±{std_ap:.4f}")
    ax.fill_between(RECALL_GRID,
                    np.clip(mean_p-std_p,0,1), np.clip(mean_p+std_p,0,1),
                    color=COLORS[mname], alpha=0.15)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Mean PR Curves ± 1 std  (N={N_REPEATS} repeats) — Elliptic Bitcoin",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1)
save_fig("robustness_01_mean_pr_curves.png")

rob_summary = repeats_df.groupby("model").agg(
    pr_auc_mean=("pr_auc","mean"), pr_auc_std=("pr_auc","std"),
    ap_mean=("ap","mean"),         ap_std=("ap","std"),
    f1_mean=("f1","mean"),         f1_std=("f1","std"),
    train_time_mean=("train_time_s","mean"),
    infer_time_mean=("infer_time_s","mean"),
).reset_index().sort_values("pr_auc_mean", ascending=False)
rob_summary.to_csv(RES_DIR / "phase3_robustness_summary.csv", index=False)
print(rob_summary[["model","pr_auc_mean","pr_auc_std","ap_mean","f1_mean","f1_std"]])

# Stability dot plot
fig, ax = plt.subplots(figsize=(9, 5))
for i, row in rob_summary.iterrows():
    mname = row["model"]
    ax.errorbar(y=mname, x=row["pr_auc_mean"], xerr=row["pr_auc_std"],
                fmt=MARKERS.get(mname,"o"), color=COLORS.get(mname,"steelblue"),
                markersize=10, capsize=6, linewidth=2, elinewidth=2,
                label=f"{mname}  {row['pr_auc_mean']:.4f}±{row['pr_auc_std']:.4f}")
ax.set_xlabel("PR-AUC (mean ± std over repeats)", fontsize=12)
ax.set_title("Model Stability: PR-AUC across Repeats — Elliptic Bitcoin",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
save_fig("robustness_02_stability_dotplot.png")

styled_table(rob_summary[["model","pr_auc_mean","pr_auc_std","ap_mean","f1_mean","f1_std","train_time_mean"]].rename(
    columns={"model":"Model","pr_auc_mean":"PR-AUC mean","pr_auc_std":"PR-AUC std",
             "ap_mean":"AP mean","f1_mean":"F1 mean","f1_std":"F1 std","train_time_mean":"Train(s)"}),
    higher_is_better=["PR-AUC mean","AP mean","F1 mean"],
    title="Robustness Summary — Mean ± Std over 5 Repeats (Elliptic Bitcoin)")

most_stable = rob_summary.loc[rob_summary["pr_auc_std"].idxmin(), "model"]
print_summary("Robustness — Elliptic Bitcoin", [
    ("Most stable model", f"{most_stable}  std={rob_summary['pr_auc_std'].min():.4f}"),
    ("Best mean PR-AUC", f"{rob_summary.iloc[0]['model']}  {rob_summary.iloc[0]['pr_auc_mean']:.4f}±{rob_summary.iloc[0]['pr_auc_std']:.4f}"),
])



5) ROBUSTNESS — REPEATED EXPERIMENTS (N=5)
  Repeat 1/5 (seed=42)...


  Repeat 2/5 (seed=43)...
  Repeat 3/5 (seed=44)...
  Repeat 4/5 (seed=45)...
  Repeat 5/5 (seed=46)...
  [saved] robustness_01_mean_pr_curves.png
                 model  pr_auc_mean  pr_auc_std   ap_mean   f1_mean    f1_std
0             LightGBM     0.984377    0.001595  0.984385  0.950114  0.002250
3              XGBoost     0.980846    0.001913  0.980856  0.942202  0.004808
2        Random Forest     0.967141    0.002900  0.967156  0.922457  0.007952
1  Logistic Regression     0.669315    0.014619  0.670533  0.718127  0.012438
  [saved] robustness_02_stability_dotplot.png


,Model,PR-AUC mean,PR-AUC std,AP mean,F1 mean,F1 std,Train(s)
0,LightGBM,0.9844,0.0016,0.9844,0.9501,0.0022,2.0356
3,XGBoost,0.9808,0.0019,0.9809,0.9422,0.0048,1.7512
2,Random Forest,0.9671,0.0029,0.9672,0.9225,0.0080,2.0324
1,Logistic Regression,0.6693,0.0146,0.6705,0.7181,0.0124,5.7039



┌────────────────────────────────────────────────────────────┐
│  📊 Robustness — Elliptic Bitcoin                          │
├────────────────────────────────────────────────────────────┤
│  Most stable model: LightGBM  std=0.0016                   │
│  Best mean PR-AUC: LightGBM  0.9844±0.0016                 │
└────────────────────────────────────────────────────────────┘


## ⚖️ 6 — Sensitivity to Class Imbalance

Same analysis as Phase 1 & 2 — training fraud prevalence varied from 0.1% to 5%.
Enables direct comparison of imbalance robustness across all three datasets.

In [9]:
# ============================================================
# 6) SENSITIVITY — IMBALANCE
# ============================================================
print("\n" + "="*70)
print("6) SENSITIVITY — IMBALANCE (all 4 models)")
print(f"   Ratios: {IMB_RATIOS}")
print("="*70)

X_tmp, X_test_imb, y_tmp, y_test_imb = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED+999)
vf = VAL_SIZE / (1.0 - TEST_SIZE)
X_pool, X_val_imb, y_pool, y_val_imb = train_test_split(
    X_tmp, y_tmp, test_size=vf, stratify=y_tmp, random_state=SEED+999)

fraud_idx = y_pool[y_pool==1].index
norm_idx  = y_pool[y_pool==0].index
imb_rows  = []

for ratio in IMB_RATIOS:
    n_fraud_t = max(1, min(int(ratio*len(norm_idx)/max(1-ratio,1e-12)), len(fraud_idx)))
    np.random.seed(SEED)
    keep = np.concatenate([norm_idx.values,
                           np.random.choice(fraud_idx, n_fraud_t, replace=False)])
    X_tr_i = X_pool.loc[keep]; y_tr_i = y_pool.loc[keep]
    tp_i = float(y_tr_i.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, SEED, tp_i, FEATURE_COLS)
        if model is None: continue
        model.fit(X_tr_i, y_tr_i)
        val_p  = model.predict_proba(X_val_imb)[:, 1]
        thr    = find_threshold_f1(y_val_imb.values, val_p)
        test_p = model.predict_proba(X_test_imb)[:, 1]
        met    = evaluate(y_test_imb.values, test_p, thr, model_name=mname)
        met.update({"target_ratio": ratio, "train_prev": tp_i})
        imb_rows.append(met)

imb_df = pd.DataFrame(imb_rows)
imb_df.to_csv(RES_DIR / "phase3_imbalance_sensitivity.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Imbalance Sensitivity — Elliptic Bitcoin", fontsize=14, fontweight="bold")
for ax, (metric, ylabel) in zip(axes, [("pr_auc","PR-AUC"),("ap","Average Precision"),("f1","F1-Score")]):
    for mname in MODEL_NAMES:
        sub = imb_df[imb_df["model"]==mname].sort_values("target_ratio")
        if sub.empty: continue
        ax.plot(sub["target_ratio"], sub[metric],
                marker=MARKERS.get(mname,"o"), color=COLORS[mname],
                linewidth=2, markersize=7, label=mname)
    ax.set_xscale("log"); ax.set_xlabel("Target Fraud Prevalence (log)")
    ax.set_ylabel(ylabel); ax.set_ylim(0,1)
axes[0].legend()
save_fig("sensitivity_01_imbalance_line.png")

pivot_imb = imb_df.pivot_table(index="model", columns="target_ratio", values="pr_auc")
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_imb, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.4, cbar_kws={"label":"PR-AUC"})
ax.set_title("PR-AUC Heatmap: Model × Imbalance Ratio — Elliptic Bitcoin", fontweight="bold")
save_fig("sensitivity_02_imbalance_heatmap.png")

imb_pivot_display = imb_df.pivot_table(index="model", columns="target_ratio", values="pr_auc").round(4).reset_index()
imb_pivot_display.columns = ["Model"] + [f"prev={c}" for c in imb_pivot_display.columns[1:]]
styled_table(imb_pivot_display, higher_is_better=[c for c in imb_pivot_display.columns if c!="Model"],
             title="PR-AUC by Target Fraud Prevalence — Elliptic Bitcoin")



6) SENSITIVITY — IMBALANCE (all 4 models)
   Ratios: [0.001, 0.005, 0.01, 0.05]
  [saved] sensitivity_01_imbalance_line.png
  [saved] sensitivity_02_imbalance_heatmap.png


,Model,prev=0.001,prev=0.005,prev=0.01,prev=0.05
0,LightGBM,0.8854,0.9550,0.9684,0.9814
1,Logistic Regression,0.5645,0.6265,0.6565,0.6953
2,Random Forest,0.9128,0.9471,0.9504,0.9636
3,XGBoost,0.9139,0.9578,0.9683,0.9798


## 📏 7 — Sensitivity to Dataset Size + Temporal Analysis

**7a.** Standard size sensitivity (same as Phase 1 & 2) — training fraction varied from 1% to 100%.

**7b.** Temporal sensitivity (unique to Phase 3) — models trained on early time steps, evaluated on later ones. This simulates a realistic deployment scenario where fraud patterns evolve over time.

In [10]:
# ============================================================
# 7) SENSITIVITY — DATASET SIZE + TEMPORAL
# ============================================================
print("\n" + "="*70)
print("7) SENSITIVITY — DATASET SIZE + TEMPORAL")
print("="*70)

# ── 7a. Standard size sensitivity ────────────────────────────
X_tmp, X_test_sz, y_tmp, y_test_sz = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED+555)
vf = VAL_SIZE / (1.0 - TEST_SIZE)
X_pool, X_val_sz, y_pool, y_val_sz = train_test_split(
    X_tmp, y_tmp, test_size=vf, stratify=y_tmp, random_state=SEED+555)

size_rows = []
for frac in SIZE_FRACS:
    if frac < 1.0:
        X_tr_s, _, y_tr_s, _ = train_test_split(
            X_pool, y_pool, train_size=frac, stratify=y_pool, random_state=SEED)
    else:
        X_tr_s, y_tr_s = X_pool, y_pool
    tp_s = float(y_tr_s.mean())

    for mname in MODEL_NAMES:
        model = build_model(mname, SEED, tp_s, FEATURE_COLS)
        if model is None: continue
        t0 = time.perf_counter()
        model.fit(X_tr_s, y_tr_s)
        tr_t = time.perf_counter() - t0
        val_p  = model.predict_proba(X_val_sz)[:, 1]
        thr    = find_threshold_f1(y_val_sz.values, val_p)
        test_p = model.predict_proba(X_test_sz)[:, 1]
        met    = evaluate(y_test_sz.values, test_p, thr, model_name=mname)
        met.update({"train_fraction": frac, "train_size": len(X_tr_s),
                    "fraud_count": int(y_tr_s.sum()), "train_time_s": tr_t})
        size_rows.append(met)

size_df = pd.DataFrame(size_rows)
size_df.to_csv(RES_DIR / "phase3_size_sensitivity.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Dataset Size Sensitivity — Elliptic Bitcoin", fontsize=14, fontweight="bold")
for ax, (metric, ylabel) in zip(axes, [("pr_auc","PR-AUC"),("f1","F1-Score"),("train_time_s","Train Time (s)")]):
    for mname in MODEL_NAMES:
        sub = size_df[size_df["model"]==mname].sort_values("train_fraction")
        if sub.empty: continue
        ax.plot(sub["train_fraction"], sub[metric],
                marker=MARKERS.get(mname,"o"), color=COLORS[mname],
                linewidth=2, markersize=7, label=mname)
    ax.set_xlabel("Training Fraction"); ax.set_ylabel(ylabel)
    if metric != "train_time_s": ax.set_ylim(0,1)
axes[0].legend()
save_fig("sensitivity_03_size_line.png")

# Learning curve: fraud count
fig, ax = plt.subplots(figsize=(9, 6))
for mname in MODEL_NAMES:
    sub = size_df[size_df["model"]==mname].sort_values("fraud_count")
    if sub.empty: continue
    ax.plot(sub["fraud_count"], sub["pr_auc"],
            marker=MARKERS.get(mname,"o"), color=COLORS[mname],
            linewidth=2, markersize=7, label=mname)
ax.set_xlabel("Number of Fraud Samples in Training Set", fontsize=12)
ax.set_ylabel("PR-AUC (Test Set)", fontsize=12)
ax.set_title("Learning Curve: PR-AUC vs Fraud Samples Available — Elliptic Bitcoin",
             fontsize=12, fontweight="bold")
ax.legend()
save_fig("sensitivity_04_learning_curve.png")

# ── 7b. Temporal sensitivity (unique to Elliptic) ────────────
print("\n  Computing temporal sensitivity...")
time_steps = sorted(df_labeled["time_step"].unique())
n_steps = len(time_steps)

# Use first 60% of time steps for training, last 40% for testing
split_idx = int(n_steps * 0.6)
train_steps = time_steps[:split_idx]
test_steps  = time_steps[split_idx:]

df_train_temp = df_labeled[df_labeled["time_step"].isin(train_steps)]
df_test_temp  = df_labeled[df_labeled["time_step"].isin(test_steps)]

X_tr_temp = df_train_temp[FEATURE_COLS]
y_tr_temp = df_train_temp["label"]
X_te_temp = df_test_temp[FEATURE_COLS]
y_te_temp = df_test_temp["label"]

print(f"  Temporal train: {len(X_tr_temp):,} rows (steps {train_steps[0]}–{train_steps[-1]})")
print(f"  Temporal test:  {len(X_te_temp):,} rows (steps {test_steps[0]}–{test_steps[-1]})")

# Split train temporally to get val set
X_tr_t2, X_va_t2, y_tr_t2, y_va_t2 = train_test_split(
    X_tr_temp, y_tr_temp, test_size=0.2, stratify=y_tr_temp, random_state=SEED)

temp_rows = []
for mname in MODEL_NAMES:
    model = build_model(mname, SEED, float(y_tr_t2.mean()), FEATURE_COLS)
    if model is None: continue
    model.fit(X_tr_t2, y_tr_t2)
    val_p  = model.predict_proba(X_va_t2)[:, 1]
    thr    = find_threshold_f1(y_va_t2.values, val_p)
    test_p = model.predict_proba(X_te_temp)[:, 1]
    met    = evaluate(y_te_temp.values, test_p, thr, model_name=mname)
    met["split"] = "temporal"
    temp_rows.append(met)

temp_df = pd.DataFrame(temp_rows)
temp_df.to_csv(RES_DIR / "phase3_temporal_sensitivity.csv", index=False)

# Compare random split vs temporal split
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(MODEL_NAMES)); w = 0.35
random_pr = [baseline_df.loc[baseline_df["model"]==m,"pr_auc"].values[0]
             if m in baseline_df["model"].values else 0 for m in MODEL_NAMES]
temporal_pr = [temp_df.loc[temp_df["model"]==m,"pr_auc"].values[0]
               if m in temp_df["model"].values else 0 for m in MODEL_NAMES]

b1 = ax.bar(x-w/2, random_pr,   width=w, label="Random split (standard)",
            color=[COLORS[m] for m in MODEL_NAMES], edgecolor="black", alpha=0.85)
b2 = ax.bar(x+w/2, temporal_pr, width=w, label="Temporal split (train t1-29, test t30-49)",
            color=[COLORS[m] for m in MODEL_NAMES], edgecolor="black", alpha=0.45,
            hatch="//")
ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, rotation=15)
ax.set_ylabel("PR-AUC"); ax.set_ylim(0, 1.1)
ax.set_title("Random vs Temporal Split — PR-AUC Comparison\n"
             "Temporal split simulates real deployment (train on past, test on future)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                    f"{h:.3f}", ha="center", fontsize=8, fontweight="bold")
save_fig("sensitivity_05_temporal_vs_random_split.png")

print_summary("Sensitivity — Elliptic Bitcoin", [
    ("Best size sensitivity model", f"{size_df[size_df['train_fraction']==1.0].set_index('model')['pr_auc'].idxmax()}"),
    ("Temporal split best model", f"{temp_df.sort_values('pr_auc',ascending=False).iloc[0]['model']}"),
    ("PR-AUC drop: random→temporal",
     f"LightGBM: {baseline_df[baseline_df['model']=='LightGBM']['pr_auc'].values[0]:.4f} → {temp_df[temp_df['model']=='LightGBM']['pr_auc'].values[0]:.4f}" if 'LightGBM' in temp_df['model'].values else "N/A"),
])



7) SENSITIVITY — DATASET SIZE + TEMPORAL
  [saved] sensitivity_03_size_line.png
  [saved] sensitivity_04_learning_curve.png

  Computing temporal sensitivity...
  Temporal train: 26,381 rows (steps 1–29)
  Temporal test:  20,183 rows (steps 30–49)
  [saved] sensitivity_05_temporal_vs_random_split.png

┌────────────────────────────────────────────────────────────┐
│  📊 Sensitivity — Elliptic Bitcoin                         │
├────────────────────────────────────────────────────────────┤
│  Best size sensitivity model: LightGBM                     │
│  Temporal split best model: LightGBM                       │
│  PR-AUC drop: random→temporal: LightGBM: 0.9848 → 0.8679   │
└────────────────────────────────────────────────────────────┘


## 🌲 8 — Feature Importance

Impurity-based importance for RF, XGBoost, LightGBM + permutation importance on best model.
Key question: do local features (node-level) or aggregated features (neighbourhood-level) drive fraud detection?

In [11]:
# ============================================================
# 8) FEATURE IMPORTANCE
# ============================================================
print("\n" + "="*70)
print("8) FEATURE IMPORTANCE")
print("="*70)

fi_models_names = ["Random Forest", "XGBoost", "LightGBM"]
feature_importances = {}

for mname in fi_models_names:
    model = build_model(mname, SEED, train_prev, FEATURE_COLS)
    if model is None: continue
    model.fit(X_train, y_train)
    clf = model.named_steps["clf"]
    if not hasattr(clf, "feature_importances_"): continue
    fi_df = (pd.DataFrame({"feature": FEATURE_COLS, "importance": clf.feature_importances_})
             .sort_values("importance", ascending=False).reset_index(drop=True))
    fi_df["rank"] = fi_df.index + 1
    fi_df["feature_type"] = fi_df["feature"].apply(
        lambda x: "local" if x.startswith("local_") else "aggregated")
    feature_importances[mname] = fi_df
    fi_df.to_csv(RES_DIR / f"phase3_fi_{mname.replace(' ','_')}.csv", index=False)
    print(f"  {mname}: top feature = {fi_df['feature'].iloc[0]} ({fi_df['importance'].iloc[0]:.4f})")

if feature_importances:
    n_fi = len(feature_importances)

    # Top 15 bars
    fig, axes = plt.subplots(1, n_fi, figsize=(7*n_fi, 8))
    fig.suptitle("Feature Importance — Top 15 per Model (Elliptic Bitcoin)",
                 fontsize=14, fontweight="bold")
    if n_fi == 1: axes = [axes]
    for ax, (mname, fi_df) in zip(axes, feature_importances.items()):
        top15 = fi_df.head(15)
        colors_fi = ["#0072B2" if t=="local" else "#D55E00"
                     for t in top15["feature_type"][::-1]]
        ax.barh(top15["feature"][::-1], top15["importance"][::-1],
                color=colors_fi, edgecolor="black", alpha=0.85)
        ax.set_title(mname, fontsize=12, fontweight="bold")
        ax.set_xlabel("Importance Score")
        for i, (feat, imp) in enumerate(zip(top15["feature"][::-1], top15["importance"][::-1])):
            ax.text(imp+0.0005, i, f"{imp:.4f}", va="center", fontsize=8)
        legend_els = [mpatches.Patch(color="#0072B2", label="Local feature"),
                      mpatches.Patch(color="#D55E00", label="Aggregated feature")]
        ax.legend(handles=legend_els, fontsize=8)
    save_fig("fi_01_top15_bars.png")

    # Local vs aggregated importance split
    fig, axes = plt.subplots(1, n_fi, figsize=(6*n_fi, 5))
    fig.suptitle("Local vs Aggregated Feature Importance — Elliptic Bitcoin\n"
                 "Shows whether node-level or neighbourhood-level features drive fraud detection",
                 fontsize=13, fontweight="bold")
    if n_fi == 1: axes = [axes]
    for ax, (mname, fi_df) in zip(axes, feature_importances.items()):
        type_imp = fi_df.groupby("feature_type")["importance"].sum()
        ax.bar(type_imp.index, type_imp.values,
               color=["#D55E00","#0072B2"], edgecolor="black", alpha=0.85)
        ax.set_title(mname, fontsize=11, fontweight="bold")
        ax.set_ylabel("Total Importance")
        for i, (ft, val) in enumerate(type_imp.items()):
            ax.text(i, val+0.005, f"{val:.3f}", ha="center", fontweight="bold")
    save_fig("fi_02_local_vs_aggregated.png")

    # Consensus heatmap
    top_n = 15
    first_rank_map = feature_importances[list(feature_importances.keys())[0]].set_index("feature")["rank"].to_dict()
    all_top_feats = sorted(
        set(f for fi_df in feature_importances.values() for f in fi_df.head(top_n)["feature"]),
        key=lambda f: first_rank_map.get(f, 99))
    consensus_matrix = pd.DataFrame(index=all_top_feats, columns=list(feature_importances.keys()))
    for mname, fi_df in feature_importances.items():
        rank_map = fi_df.set_index("feature")["rank"].to_dict()
        for feat in all_top_feats:
            consensus_matrix.loc[feat, mname] = rank_map.get(feat, np.nan)
    consensus_matrix = consensus_matrix.astype(float)

    fig, ax = plt.subplots(figsize=(8, max(6, len(all_top_feats)*0.4)))
    sns.heatmap(consensus_matrix, annot=True, fmt=".0f", cmap="YlOrRd_r",
                linewidths=0.4, ax=ax,
                cbar_kws={"label":"Feature Rank (lower=more important)"},
                vmin=1, vmax=top_n)
    ax.set_title(f"Feature Importance Consensus (Rank, Top {top_n} per model)",
                 fontweight="bold")
    ax.set_xlabel("Model"); ax.set_ylabel("Feature")
    save_fig("fi_03_consensus_heatmap.png")

    # Permutation importance
    from sklearn.inspection import permutation_importance as sklearn_perm
    perm_model_name = next((m for m in ["LightGBM","XGBoost","Random Forest"]
                            if m in feature_importances), None)
    if perm_model_name:
        print(f"\n  Computing permutation importance for {perm_model_name}...")
        clf_perm = build_model(perm_model_name, SEED, train_prev, FEATURE_COLS)
        clf_perm.fit(X_train, y_train)

        rng_perm = np.random.default_rng(SEED)
        perm_idx = rng_perm.choice(len(X_val), min(5000, len(X_val)), replace=False)
        X_val_arr = X_val.iloc[perm_idx].values
        y_val_arr = y_val.iloc[perm_idx].values

        pi = sklearn_perm(clf_perm, X_val_arr, y_val_arr,
                          scoring="average_precision", n_repeats=10,
                          random_state=SEED, n_jobs=-1)
        perm_df = (pd.DataFrame({
            "feature": FEATURE_COLS,
            "importance_mean": pi.importances_mean,
            "importance_std":  pi.importances_std,
        }).sort_values("importance_mean", ascending=False).reset_index(drop=True))
        perm_df["feature_type"] = perm_df["feature"].apply(
            lambda x: "local" if x.startswith("local_") else "aggregated")
        perm_df.to_csv(RES_DIR / "phase3_fi_permutation.csv", index=False)

        top15_p = perm_df.head(15)
        colors_p = ["#0072B2" if t=="local" else "#D55E00"
                    for t in top15_p["feature_type"][::-1]]
        fig, ax = plt.subplots(figsize=(11, 7))
        ax.barh(top15_p["feature"][::-1], top15_p["importance_mean"][::-1],
                xerr=top15_p["importance_std"][::-1],
                color=colors_p, edgecolor="black", alpha=0.85, capsize=4)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("Mean decrease in AP when feature is permuted")
        ax.set_title(f"Permutation Importance — {perm_model_name} (Elliptic Bitcoin)\n"
                     "Blue = local feature | Orange = aggregated feature",
                     fontsize=11, fontweight="bold")
        for i, (m, s) in enumerate(zip(top15_p["importance_mean"][::-1],
                                       top15_p["importance_std"][::-1])):
            ax.text(max(m,0)+0.001, i, f"{m:.4f}±{s:.4f}", va="center", fontsize=8)
        save_fig("fi_04_permutation_importance.png")
        print(f"  Top permutation feature: {perm_df['feature'].iloc[0]}")
        print(f"  Top feature type: {perm_df['feature_type'].iloc[0]}")

    # Consensus table
    votes = Counter([f for fi_df in feature_importances.values() for f in fi_df.head(10)["feature"]])
    consensus_df = (pd.DataFrame([(f, c) for f, c in votes.items() if c >= 2],
                                  columns=["feature","models_voting"])
                    .sort_values("models_voting", ascending=False))
    consensus_df["feature_type"] = consensus_df["feature"].apply(
        lambda x: "local" if x.startswith("local_") else "aggregated")
    consensus_df.to_csv(RES_DIR / "phase3_fi_consensus.csv", index=False)
    styled_table(consensus_df.head(12),
                 higher_is_better=["models_voting"],
                 title="Feature Importance Consensus — Elliptic Bitcoin (features in ≥2 models' top-10)")

    print_summary("Feature Importance — Elliptic Bitcoin", [
        ("Top impurity feature", consensus_df.iloc[0]["feature"] if not consensus_df.empty else "N/A"),
        ("Top permutation feature", perm_df["feature"].iloc[0] if "perm_df" in dir() else "N/A"),
        ("Dominant feature type",
         perm_df.head(5)["feature_type"].mode()[0] if "perm_df" in dir() else "N/A"),
        ("Features in all 3 models' top-10", str((consensus_df["models_voting"]==3).sum())),
    ])



8) FEATURE IMPORTANCE
  Random Forest: top feature = local_f55 (0.0631)
  XGBoost: top feature = local_f14 (0.0766)
  LightGBM: top feature = local_f2 (251.0000)
  [saved] fi_01_top15_bars.png
  [saved] fi_02_local_vs_aggregated.png
  [saved] fi_03_consensus_heatmap.png

  Computing permutation importance for LightGBM...
  [saved] fi_04_permutation_importance.png
  Top permutation feature: local_f59
  Top feature type: local


,feature,models_voting,feature_type
4,local_f47,3,local
5,local_f53,3,local
0,local_f55,2,local
1,local_f14,2,local
2,local_f90,2,local
3,local_f5,2,local
6,local_f49,2,local



┌────────────────────────────────────────────────────────────┐
│  📊 Feature Importance — Elliptic Bitcoin                  │
├────────────────────────────────────────────────────────────┤
│  Top impurity feature: local_f47                           │
│  Top permutation feature: local_f59                        │
│  Dominant feature type: local                              │
│  Features in all 3 models' top-10: 2                       │
└────────────────────────────────────────────────────────────┘


## 📐 9 — Probability Calibration

Brier Score + ECE + Reliability Diagrams — consistent with Phase 1 & 2.

In [12]:
# ============================================================
# 9) CALIBRATION ANALYSIS
# ============================================================
print("\n" + "="*70)
print("9) CALIBRATION ANALYSIS")
print("="*70)

y_test_np = y_test.values
calib_stats = {}

for mname, proba in pr_cache.items():
    brier = brier_score_loss(y_test_np, proba)
    ece   = compute_ece(y_test_np, proba, n_bins=10)
    calib_stats[mname] = {"brier": brier, "ece": ece}
    print(f"  {mname}: Brier={brier:.4f}  ECE={ece:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Probability Calibration — Elliptic Bitcoin\n"
             "Reliability diagram: does predicted P(fraud) match observed fraud rate?\n"
             "Perfect calibration = diagonal line",
             fontsize=12, fontweight="bold")

ax_cal = axes[0]; ax_hist = axes[1]
ax_cal.plot([0,1],[0,1],"k--",linewidth=1.5, label="Perfect calibration", zorder=0)
for mname, proba in pr_cache.items():
    frac_pos, mean_pred = calibration_curve(y_test_np, proba, n_bins=12, strategy="quantile")
    ece = calib_stats[mname]["ece"]
    ax_cal.plot(mean_pred, frac_pos, marker="o", markersize=5, linewidth=2,
                color=COLORS[mname], label=f"{mname}  ECE={ece:.3f}")
ax_cal.set_xlabel("Mean Predicted Probability"); ax_cal.set_ylabel("Observed Fraud Rate")
ax_cal.set_title("Reliability Diagram", fontweight="bold")
ax_cal.legend(fontsize=9); ax_cal.set_xlim(0,1); ax_cal.set_ylim(0,1)

for mname, proba in pr_cache.items():
    ax_hist.hist(proba, bins=50, alpha=0.45, color=COLORS[mname], label=mname, density=True)
ax_hist.set_xlabel("Predicted P(fraud)"); ax_hist.set_ylabel("Density")
ax_hist.set_title("Distribution of Predicted Probabilities", fontweight="bold")
ax_hist.legend(fontsize=9); ax_hist.set_xlim(0,1)
save_fig("calibration_01_reliability_diagram.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Calibration Quality Summary — Elliptic Bitcoin\n"
             "Brier Score and ECE: lower is better",
             fontsize=11, fontweight="bold")
for ax, (metric_key, ylabel) in zip(axes, [("brier","Brier Score"),("ece","ECE")]):
    names  = [m for m in MODEL_NAMES if m in calib_stats]
    values = [calib_stats[m][metric_key] for m in names]
    colors = [COLORS[m] for m in names]
    bars = ax.bar(names, values, color=colors, edgecolor="black", alpha=0.85)
    ax.set_ylabel(ylabel); ax.set_title(ylabel, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(0, max(values)*1.35)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(values)*0.02,
                f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")
    if metric_key == "brier":
        naive = brier_score_loss(y_test_np, np.full_like(y_test_np, y_test_np.mean(), dtype=float))
        ax.axhline(naive, color="gray", linestyle="--", linewidth=1.5,
                   label=f"Naive baseline: {naive:.4f}")
        ax.legend(fontsize=8)
save_fig("calibration_02_scores.png")

pd.DataFrame([{"model":m,"brier_score":v["brier"],"ece":v["ece"]}
              for m,v in calib_stats.items()]).to_csv(
    RES_DIR / "phase3_calibration_stats.csv", index=False)

calib_df = pd.DataFrame([{"Model":m,"Brier Score":v["brier"],"ECE":v["ece"]}
                          for m,v in calib_stats.items()]).sort_values("Brier Score")
styled_table(calib_df, title="Calibration Quality — Elliptic Bitcoin (lower is better for both)")

best_cal = calib_df.iloc[0]
print_summary("Calibration — Elliptic Bitcoin", [
    ("Best calibrated model", f"{best_cal['Model']}  Brier={best_cal['Brier Score']:.4f}  ECE={best_cal['ECE']:.4f}"),
    ("Worst calibrated model", f"{calib_df.iloc[-1]['Model']}  Brier={calib_df.iloc[-1]['Brier Score']:.4f}"),
])



9) CALIBRATION ANALYSIS
  Logistic Regression: Brier=0.0975  ECE=0.1443
  Random Forest: Brier=0.0209  ECE=0.0593
  XGBoost: Brier=0.0138  ECE=0.0254
  LightGBM: Brier=0.0094  ECE=0.0143
  [saved] calibration_01_reliability_diagram.png
  [saved] calibration_02_scores.png


,Model,Brier Score,ECE
3,LightGBM,0.0094,0.0143
2,XGBoost,0.0138,0.0254
1,Random Forest,0.0209,0.0593
0,Logistic Regression,0.0975,0.1443



┌────────────────────────────────────────────────────────────┐
│  📊 Calibration — Elliptic Bitcoin                         │
├────────────────────────────────────────────────────────────┤
│  Best calibrated model: LightGBM  Brier=0.0094  ECE=0.0143 │
│  Worst calibrated model: Logistic Regression  Brier=0.0975 │
└────────────────────────────────────────────────────────────┘


## 📊 10 — Statistical Significance Tests

Paired t-tests and Wilcoxon signed-rank tests on PR-AUC across 5 repeats.

In [13]:
# ============================================================
# 10) STATISTICAL SIGNIFICANCE TESTS
# ============================================================
print("\n" + "="*70)
print("10) STATISTICAL SIGNIFICANCE TESTS")
print(f"    Metric: PR-AUC across {N_REPEATS} repeats")
print("="*70)

available_models = [m for m in MODEL_NAMES if m in repeats_df["model"].unique()]
sig_rows = []

for m1, m2 in combinations(available_models, 2):
    vals_m1 = repeats_df[repeats_df["model"]==m1].sort_values("repeat")["pr_auc"].values
    vals_m2 = repeats_df[repeats_df["model"]==m2].sort_values("repeat")["pr_auc"].values

    if len(vals_m1) >= 2 and len(vals_m2) >= 2:
        n = min(len(vals_m1), len(vals_m2))
        t_stat, t_pval = ttest_rel(vals_m1[:n], vals_m2[:n])
        try:
            w_stat, w_pval = wilcoxon(vals_m1[:n], vals_m2[:n])
        except Exception:
            w_stat, w_pval = np.nan, np.nan

        mean_diff = float(np.mean(vals_m1[:n]) - np.mean(vals_m2[:n]))
        sig_rows.append({
            "model_A": m1, "model_B": m2,
            "mean_A": float(np.mean(vals_m1[:n])),
            "mean_B": float(np.mean(vals_m2[:n])),
            "mean_diff (A-B)": mean_diff,
            "t_statistic": float(t_stat),
            "t_pvalue": float(t_pval),
            "wilcoxon_stat": float(w_stat) if not np.isnan(w_stat) else None,
            "wilcoxon_pvalue": float(w_pval) if not np.isnan(w_pval) else None,
            "significant_p05": bool(t_pval < 0.05),
        })
        marker = "*" if t_pval < 0.05 else " "
        print(f"  {m1} vs {m2}: diff={mean_diff:+.4f}  t-p={t_pval:.4f}  {marker}")

sig_df = pd.DataFrame(sig_rows)
# ── Bonferroni correction ─────────────────────────────────────
# With C(4,2)=6 pairwise comparisons, the familywise error rate
# at α=0.05 inflates. Bonferroni-corrected threshold: α/6 ≈ 0.0083.
n_comparisons = len(sig_rows)
alpha_bonferroni = 0.05 / max(n_comparisons, 1)
print(f"\n  Bonferroni-corrected α: 0.05 / {n_comparisons} = {alpha_bonferroni:.4f}")
print(f"  Pairs significant after Bonferroni: "
      f"{sum(r['t_pvalue'] < alpha_bonferroni for r in sig_rows)} / {n_comparisons}")
print(f"  Note: N={N_REPEATS} repeats — interpret as indicative, not conclusive.")
if not sig_df.empty:
    sig_df["bonferroni_sig"] = sig_df["t_pvalue"] < alpha_bonferroni
sig_df.to_csv(RES_DIR / "phase3_significance_tests.csv", index=False)

n_m = len(available_models)
pval_matrix = pd.DataFrame(np.ones((n_m, n_m)),
                            index=available_models, columns=available_models)
for row in sig_rows:
    pval_matrix.loc[row["model_A"], row["model_B"]] = row["t_pvalue"]
    pval_matrix.loc[row["model_B"], row["model_A"]] = row["t_pvalue"]
# FIX: extract writable numpy copy before fill_diagonal
arr = pval_matrix.values.copy()
np.fill_diagonal(arr, 1.0)
pval_matrix = pd.DataFrame(arr, index=available_models, columns=available_models)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f",
            cmap="RdYlGn_r", ax=ax, vmin=0, vmax=0.1,
            linewidths=0.5, cbar_kws={"label":"Paired t-test p-value"})
ax.set_title(f"Pairwise Significance Tests — PR-AUC ({N_REPEATS} repeats) — Elliptic Bitcoin\n"
             "Green = significant difference (p<0.05), Red = not significant\n"
             f"Note: N={N_REPEATS} repeats; interpret with caution (low power)",
             fontsize=10, fontweight="bold")
save_fig("significance_01_pairwise_heatmap.png")

if not sig_df.empty:
    sig_display = sig_df[["model_A","model_B","mean_A","mean_B",
                           "mean_diff (A-B)","t_pvalue","wilcoxon_pvalue","significant_p05"]].copy()
    sig_display.columns = ["Model A","Model B","Mean A","Mean B",
                            "Diff (A-B)","t p-value","Wilcoxon p-value","Sig. p<0.05"]
    styled_table(sig_display, title="Pairwise Statistical Tests — Elliptic Bitcoin")

    n_sig = sig_df["significant_p05"].sum()
    summary_rows_sig = [("Significant pairs (p<0.05)", f"{n_sig} / {len(sig_df)}")]
    for _, r in sig_df.iterrows():
        summary_rows_sig.append((f"{r['model_A']} vs {r['model_B']} p-value", f"{r['t_pvalue']:.4f}"))
    summary_rows_sig.append(("Key finding", "Pairs marked * are significant at p<0.05"))
    print_summary("Significance Tests — Elliptic Bitcoin", summary_rows_sig)



10) STATISTICAL SIGNIFICANCE TESTS
    Metric: PR-AUC across 5 repeats
  Logistic Regression vs Random Forest: diff=-0.2978  t-p=0.0000  *
  Logistic Regression vs XGBoost: diff=-0.3115  t-p=0.0000  *
  Logistic Regression vs LightGBM: diff=-0.3151  t-p=0.0000  *
  Random Forest vs XGBoost: diff=-0.0137  t-p=0.0001  *
  Random Forest vs LightGBM: diff=-0.0172  t-p=0.0000  *
  XGBoost vs LightGBM: diff=-0.0035  t-p=0.0012  *

  Bonferroni-corrected α: 0.05 / 6 = 0.0083
  Pairs significant after Bonferroni: 6 / 6
  Note: N=5 repeats — interpret as indicative, not conclusive.
  [saved] significance_01_pairwise_heatmap.png


,Model A,Model B,Mean A,Mean B,Diff (A-B),t p-value,Wilcoxon p-value,Sig. p<0.05
0,Logistic Regression,Random Forest,0.6693,0.9671,-0.2978,0.0000,0.0625,True
1,Logistic Regression,XGBoost,0.6693,0.9808,-0.3115,0.0000,0.0625,True
2,Logistic Regression,LightGBM,0.6693,0.9844,-0.3151,0.0000,0.0625,True
3,Random Forest,XGBoost,0.9671,0.9808,-0.0137,0.0001,0.0625,True
4,Random Forest,LightGBM,0.9671,0.9844,-0.0172,0.0000,0.0625,True
5,XGBoost,LightGBM,0.9808,0.9844,-0.0035,0.0012,0.0625,True



┌────────────────────────────────────────────────────────────┐
│  📊 Significance Tests — Elliptic Bitcoin                  │
├────────────────────────────────────────────────────────────┤
│  Significant pairs (p<0.05): 6 / 6                         │
│  Logistic Regression vs Random Forest p-value: 0.0000      │
│  Logistic Regression vs XGBoost p-value: 0.0000            │
│  Logistic Regression vs LightGBM p-value: 0.0000           │
│  Random Forest vs XGBoost p-value: 0.0001                  │
│  Random Forest vs LightGBM p-value: 0.0000                 │
│  XGBoost vs LightGBM p-value: 0.0012                       │
│  Key finding: Pairs marked * are significant at p<0.05     │
└────────────────────────────────────────────────────────────┘


## 🔍 11 — SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) values are computed for **LightGBM** — the best or tied-best model across all phases.

Unlike impurity-based importance, SHAP quantifies each feature's **directional contribution** to individual predictions, grounded in cooperative game theory.

**Three outputs:**
- **Beeswarm plot** — global feature importance with direction (red = high value pushing toward fraud, blue = low value)
- **Bar plot** — mean |SHAP| per feature (magnitude-only global ranking)
- **Dependence plot** — how the most important feature's SHAP value varies with its raw value

> SHAP values are computed on a stratified subsample of the test set (up to 2,000 rows) for computational efficiency.

In [14]:
# ============================================================
# 11) SHAP EXPLAINABILITY — LightGBM (Phase 3)
# ============================================================
print("\n" + "="*70)
print("11) SHAP ANALYSIS — LightGBM")
print("="*70)

try:
    import shap
    shap_available = True
except ImportError:
    print("WARNING: shap not installed.  Run:  pip install shap")
    shap_available = False

if shap_available and "LightGBM" in model_cache:
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")

    lgbm_pipeline      = model_cache["LightGBM"]
    clf_shap           = lgbm_pipeline.named_steps["clf"]
    prep_shap          = lgbm_pipeline.named_steps["prep"]
    Xp_test_shap       = prep_shap.transform(X_test)
    feature_names_shap = list(FEATURE_COLS)

    np.random.seed(SEED)
    n_shap   = min(2000, len(X_test))
    idx_shap = np.random.choice(len(X_test), n_shap, replace=False)
    Xp_shap_sample = Xp_test_shap[idx_shap]

    print(f"  Computing SHAP values on {n_shap} test samples...")
    explainer = shap.TreeExplainer(clf_shap)
    shap_raw  = explainer.shap_values(Xp_shap_sample)

    if isinstance(shap_raw, list):
        sv = np.array(shap_raw[1])
    elif shap_raw.ndim == 3:
        sv = shap_raw[:, :, 1]
    else:
        sv = shap_raw

    Xp_shap_df    = pd.DataFrame(Xp_shap_sample, columns=feature_names_shap)
    mean_abs_shap = np.abs(sv).mean(axis=0)

    # ── 11a. Beeswarm ────────────────────────────────────────
    shap.summary_plot(sv, Xp_shap_df, max_display=15, show=False)
    fig = plt.gcf()
    fig.suptitle("SHAP Beeswarm — LightGBM (Phase 3: Elliptic Bitcoin)\n"
                 "Red = high value → illicit  |  Blue = low value",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_fig("shap_01_beeswarm.png")

    # ── 11b. Bar ─────────────────────────────────────────────
    shap.summary_plot(sv, Xp_shap_df, plot_type="bar", max_display=15, show=False)
    fig = plt.gcf()
    fig.suptitle("SHAP Mean |SHAP| — LightGBM (Phase 3)",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_fig("shap_02_bar.png")

    # ── 11c. Dependence plot ─────────────────────────────────
    top_feat_idx  = int(np.argmax(mean_abs_shap))
    top_feat_name = feature_names_shap[top_feat_idx]
    print(f"  Top SHAP feature: {top_feat_name}")

    shap.dependence_plot(top_feat_idx, sv, Xp_shap_df,
                         feature_names=feature_names_shap, show=False)
    fig = plt.gcf()
    fig.suptitle(f"SHAP Dependence — {top_feat_name} (Phase 3)\n"
                 "How this feature drives illicit classification",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_fig("shap_03_dependence.png")

    # ── 11d. Local vs aggregated SHAP breakdown ───────────────
    local_mask = [f.startswith("local_") for f in feature_names_shap]
    agg_mask   = [f.startswith("agg_")   for f in feature_names_shap]
    local_total = float(np.abs(sv[:, local_mask]).mean())
    agg_total   = float(np.abs(sv[:, agg_mask]).mean())
    grand_total = local_total + agg_total

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(["Local features", "Aggregated features"],
           [local_total, agg_total],
           color=["#0072B2", "#E69F00"], edgecolor="black", alpha=0.85)
    ax.set_ylabel("Mean |SHAP| value", fontsize=11)
    ax.set_title("SHAP: Local vs Aggregated Feature Contribution\n"
                 "LightGBM — Phase 3 (Elliptic Bitcoin)",
                 fontsize=12, fontweight="bold")
    for i, v in enumerate([local_total, agg_total]):
        ax.text(i, v*1.02, f"{v:.4f}\n({v/grand_total*100:.1f}%)",
                ha="center", fontsize=11)
    save_fig("shap_04_local_vs_agg.png")

    # ── 11e. CSV ─────────────────────────────────────────────
    shap_df = pd.DataFrame({
        "feature":       feature_names_shap[:sv.shape[1]],
        "mean_abs_shap": mean_abs_shap,
        "mean_shap":     sv.mean(axis=0),
        "feature_type":  ["local" if f.startswith("local_") else
                          "aggregated" if f.startswith("agg_") else "other"
                          for f in feature_names_shap[:sv.shape[1]]],
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_df["rank"] = shap_df.index + 1
    shap_df.to_csv(RES_DIR / "phase3_shap_importance.csv", index=False)
    print("\nTop 10 features by mean |SHAP|:")
    print(shap_df[["rank","feature","feature_type","mean_abs_shap","mean_shap"]].head(10).to_string(index=False))

    print_summary("SHAP Analysis — LightGBM Phase 3", [
        ("Top SHAP feature",              top_feat_name),
        ("Mean |SHAP| top feature",        f"{mean_abs_shap[top_feat_idx]:.4f}"),
        ("Local features SHAP share",      f"{local_total:.4f} ({local_total/grand_total*100:.1f}%)"),
        ("Aggregated features SHAP share",  f"{agg_total:.4f} ({agg_total/grand_total*100:.1f}%)"),
        ("Samples used",                   str(n_shap)),
    ])
else:
    print("  SHAP not available or LightGBM not in cache — skipping.")



11) SHAP ANALYSIS — LightGBM
  Computing SHAP values on 2000 test samples...
  [saved] shap_01_beeswarm.png
  [saved] shap_02_bar.png
  Top SHAP feature: local_f59
  [saved] shap_03_dependence.png
  [saved] shap_04_local_vs_agg.png

Top 10 features by mean |SHAP|:
 rank   feature feature_type  mean_abs_shap  mean_shap
    1 local_f59        local       1.657171  -0.291795
    2 local_f53        local       0.510099  -0.117581
    3  local_f5        local       0.293429   0.021962
    4 local_f76        local       0.239517   0.048676
    5 local_f90        local       0.215576   0.006180
    6 local_f58        local       0.210845   0.008979
    7 local_f46        local       0.181990   0.118056
    8 local_f16        local       0.153916   0.006782
    9 local_f55        local       0.147047  -0.032719
   10 local_f65        local       0.132829   0.095250

┌────────────────────────────────────────────────────────────┐
│  📊 SHAP Analysis — LightGBM Phase 3                       │
├──

## 📋 12 — Summary Dashboard

In [15]:
# ============================================================
# 12) SUMMARY DASHBOARD
# ============================================================
print("\n" + "="*70)
print("12) SUMMARY DASHBOARD")
print("="*70)

fig = plt.figure(figsize=(20, 26))
fig.suptitle("GP Phase 3 — Summary Dashboard\nDataset: Elliptic Bitcoin Transactions",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.35)

# Row 0: Class distribution + config + temporal
ax_pie  = fig.add_subplot(gs[0, 0])
ax_text = fig.add_subplot(gs[0, 1])
ax_time = fig.add_subplot(gs[0, 2])

counts_vals = [y.value_counts().get(0,0), y.value_counts().get(1,0)]
ax_pie.pie(counts_vals, labels=["Licit","Illicit"],
           colors=["#0072B2","#D55E00"],
           autopct=lambda p: f"{p:.2f}%", explode=[0,0.08], startangle=90)
ax_pie.set_title("Class Distribution", fontweight="bold")

info_text = (
    f"Dataset: Elliptic Bitcoin Transactions\n"
    f"Total rows (labeled): {len(df_labeled):,}\n"
    f"Features: {len(FEATURE_COLS)} (94 local + 72 agg.)\n"
    f"Fraud cases: {y.sum():,} ({y.mean():.4%})\n"
    f"Imbalance: {(y==0).sum()//(y==1).sum()}:1\n"
    f"Time steps: {df_labeled['time_step'].nunique()}\n\n"
    f"Split: 70% Train / 15% Val / 15% Test\n"
    f"Repeats: N={N_REPEATS}  Bootstrap: N={N_BOOTSTRAP}\n"
    f"Imbalance strategy: class_weight=balanced"
)
ax_text.text(0.05, 0.5, info_text, transform=ax_text.transAxes,
             fontsize=9.5, verticalalignment="center",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
ax_text.axis("off"); ax_text.set_title("Experiment Configuration", fontweight="bold")

if "time_stats" in dir():
    ax_time.plot(time_stats["time_step"], time_stats["fraud_rate"],
                 color="#D55E00", linewidth=2, marker="o", markersize=3)
    ax_time.axhline(y.mean()*100, color="gray", linestyle="--", linewidth=1)
    ax_time.set_xlabel("Time Step"); ax_time.set_ylabel("Fraud Rate (%)")
    ax_time.set_title("Fraud Rate per Time Step", fontweight="bold")

# Row 1: PR + ROC + best model CM
ax_pr  = fig.add_subplot(gs[1, 0])
ax_roc = fig.add_subplot(gs[1, 1])
ax_cm  = fig.add_subplot(gs[1, 2])

ax_pr.hlines(float(y_test.mean()), 0, 1, linestyles="--", colors="gray", linewidth=1.2)
for mname, proba in pr_cache.items():
    p, r, _ = precision_recall_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    ap = average_precision_score(y_test.values, proba)
    ax_pr.plot(r, p, color=COLORS[mname], linewidth=1.5, label=f"{mname[:2]}. AP={ap:.3f}")
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.set_title("PR Curves (Test)", fontweight="bold")
ax_pr.legend(fontsize=7); ax_pr.set_xlim(0,1); ax_pr.set_ylim(0,1)

ax_roc.plot([0,1],[0,1],"k--", linewidth=1)
for mname, proba in pr_cache.items():
    fp_r, tp_r, _ = roc_curve(y_test.values, proba, pos_label=FRAUD_LABEL)
    roc = roc_auc_score(y_test.values, proba)
    ax_roc.plot(fp_r, tp_r, color=COLORS[mname], linewidth=1.5, label=f"{mname[:2]}. {roc:.3f}")
ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_title("ROC Curves (Test)", fontweight="bold")
ax_roc.legend(fontsize=7)

best_model_name = baseline_df.iloc[0]["model"]
proba_best = pr_cache.get(best_model_name)
if proba_best is not None:
    thr_best = thr_cache[best_model_name]
    y_pred_best = (proba_best >= thr_best).astype(int)
    cm_best = confusion_matrix(y_test.values, y_pred_best, labels=[NORMAL_LABEL, FRAUD_LABEL])
    sns.heatmap(cm_best, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Licit","Illicit"], yticklabels=["Licit","Illicit"],
                cbar=False, ax=ax_cm, annot_kws={"size":11})
    ax_cm.set_xlabel("Predicted"); ax_cm.set_ylabel("Actual")
    ax_cm.set_title(f"Best Model CM\n({best_model_name})", fontweight="bold")

# Row 2: Performance bars + Calibration
ax_bar   = fig.add_subplot(gs[2, 0:2])
ax_calib = fig.add_subplot(gs[2, 2])

metrics_show = ["pr_auc","ap","f1","recall","precision"]
x = np.arange(len(metrics_show))
w = 0.8 / max(len(baseline_df), 1)
for i, (_, row) in enumerate(baseline_df.iterrows()):
    vals = [row.get(m,0) for m in metrics_show]
    offset = (i - len(baseline_df)/2 + 0.5) * w
    ax_bar.bar(x+offset, vals, width=w*0.85,
               color=COLORS.get(row["model"],"steelblue"),
               edgecolor="black", alpha=0.85, label=row["model"])
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(["PR-AUC","Avg Prec","F1","Recall","Precision"], fontsize=10)
ax_bar.set_ylim(0, 1.15); ax_bar.set_ylabel("Score")
ax_bar.set_title("Model Performance Comparison (Test Set)", fontweight="bold")
ax_bar.legend(fontsize=8, loc="upper right")

if calib_stats:
    calib_names = [m for m in MODEL_NAMES if m in calib_stats]
    brier_vals  = [calib_stats[m]["brier"] for m in calib_names]
    ax_calib.bar(calib_names, brier_vals,
                 color=[COLORS.get(m,"gray") for m in calib_names],
                 edgecolor="black", alpha=0.85)
    ax_calib.set_title("Brier Score (↓ better)", fontweight="bold")
    ax_calib.set_ylabel("Brier Score")
    ax_calib.tick_params(axis="x", rotation=20)
    for i, (name, val) in enumerate(zip(calib_names, brier_vals)):
        ax_calib.text(i, val+max(brier_vals)*0.02, f"{val:.4f}",
                      ha="center", fontsize=8, fontweight="bold")

# Row 3: Imbalance + Size + Feature importance
ax_imb = fig.add_subplot(gs[3, 0])
ax_sz  = fig.add_subplot(gs[3, 1])
ax_fi  = fig.add_subplot(gs[3, 2])

for mname in MODEL_NAMES:
    sub = imb_df[imb_df["model"]==mname].sort_values("target_ratio")
    if sub.empty: continue
    ax_imb.plot(sub["target_ratio"], sub["pr_auc"],
                marker=MARKERS.get(mname,"o"), color=COLORS[mname],
                linewidth=1.5, markersize=5, label=mname[:5])
ax_imb.set_xscale("log"); ax_imb.set_xlabel("Fraud Prevalence (log)")
ax_imb.set_ylabel("PR-AUC"); ax_imb.set_ylim(0,1)
ax_imb.set_title("Imbalance Sensitivity", fontweight="bold")
ax_imb.legend(fontsize=7)

for mname in MODEL_NAMES:
    sub = size_df[size_df["model"]==mname].sort_values("train_fraction")
    if sub.empty: continue
    ax_sz.plot(sub["train_fraction"], sub["pr_auc"],
               marker=MARKERS.get(mname,"o"), color=COLORS[mname],
               linewidth=1.5, markersize=5, label=mname[:5])
ax_sz.set_xlabel("Training Fraction"); ax_sz.set_ylabel("PR-AUC")
ax_sz.set_title("Size Sensitivity", fontweight="bold")
ax_sz.legend(fontsize=7); ax_sz.set_ylim(0,1)

if feature_importances:
    fi_best = feature_importances[list(feature_importances.keys())[0]].head(10)
    colors_fi_dash = ["#0072B2" if t=="local" else "#D55E00"
                      for t in fi_best["feature_type"][::-1]]
    ax_fi.barh(fi_best["feature"][::-1], fi_best["importance"][::-1],
               color=colors_fi_dash, edgecolor="black", alpha=0.85)
    ax_fi.set_xlabel("Importance")
    ax_fi.set_title(f"Top 10 Features\n({list(feature_importances.keys())[0]})",
                    fontweight="bold")
else:
    ax_fi.axis("off")

# Row 4: Timing + significance
ax_tim = fig.add_subplot(gs[4, 0])
ax_sig = fig.add_subplot(gs[4, 1:3])

sub_tim = baseline_df.set_index("model").reindex(
    [m for m in MODEL_NAMES if m in baseline_df["model"].values])
ax_tim.barh(sub_tim.index, sub_tim["train_time_s"],
            color=[COLORS.get(m,"gray") for m in sub_tim.index],
            edgecolor="black", alpha=0.85)
ax_tim.set_xlabel("Training Time (s)")
ax_tim.set_title("Training Time Comparison", fontweight="bold")
for i, (idx, row) in enumerate(sub_tim.iterrows()):
    ax_tim.text(row["train_time_s"]*1.02, i, f"{row['train_time_s']:.1f}s",
                va="center", fontsize=9)

if not pval_matrix.empty:
    sns.heatmap(pval_matrix.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", ax=ax_sig, vmin=0, vmax=0.1,
                linewidths=0.5, cbar_kws={"label":"p-value"})
    ax_sig.set_title(f"Pairwise Significance (paired t-test, N={N_REPEATS} repeats)\nGreen = significant (p<0.05)",
                     fontweight="bold", fontsize=9)

save_fig("SUMMARY_DASHBOARD.png", tight=False)



12) SUMMARY DASHBOARD
  [saved] SUMMARY_DASHBOARD.png


## 🔀 13 — 3-Way Cross-Phase Comparison

The central contribution of Phase 3: extending the cross-phase comparison to three datasets with distinct characteristics:

| | Phase 1 (Kaggle CC) | Phase 2 (IEEE-CIS) | Phase 3 (Elliptic Bitcoin) |
|---|---|---|---|
| Domain | Banking card | E-commerce | Cryptocurrency |
| Fraud rate | 0.17% | 3.5% | ~9.8% |
| Features | 30 (PCA) | 224 (mixed) | 166 (local+agg) |
| Structure | Tabular | Tabular + identity | Graph-based (tabular) |
| Imbalance | Extreme | Moderate | Low-moderate |

In [16]:
# ============================================================
# 13) 3-WAY CROSS-PHASE COMPARISON
# ============================================================
print("\n" + "="*70)
print("13) 3-WAY CROSS-PHASE COMPARISON")
print("="*70)

# Patched styled_table — handles NaN values safely
def styled_table(df, higher_is_better=None, title=""):
    if higher_is_better is None: higher_is_better = []
    def color_col(s, hib=True):
        numeric_vals = []
        for v in s:
            try:
                numeric_vals.append(float(v))
            except (ValueError, TypeError):
                numeric_vals.append(np.nan)
        valid = [v for v in numeric_vals if not np.isnan(v)]
        if not valid:
            return [""] * len(s)
        vmin, vmax = min(valid), max(valid)
        rng = vmax - vmin if vmax != vmin else 1
        colors = []
        for v in numeric_vals:
            if np.isnan(v):
                colors.append("")
                continue
            norm = (v - vmin) / rng
            if hib: r, g, b = int(255*(1-norm)), int(200*norm+55), 60
            else:   r, g, b = int(200*norm+55), int(255*(1-norm)), 60
            colors.append(f"background-color: rgba({r},{g},{b},0.25); font-weight: bold")
        return colors
    styler = df.style
    for col in df.columns:
        if col in higher_is_better:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=True), subset=[col])
        elif df[col].dtype in ["float64","float32"] and col not in ["threshold"]:
            styler = styler.apply(lambda s, c=col: color_col(s, hib=False), subset=[col])
    styler = (styler
        .set_caption(f"<b>{title}</b>")
        .set_table_styles([
            {"selector":"caption","props":[("font-size","15px"),("text-align","left"),("padding","8px 0"),("color","#222")]},
            {"selector":"th","props":[("background","#2C3E50"),("color","white"),("padding","8px 12px"),("font-size","11px")]},
            {"selector":"td","props":[("padding","6px 12px"),("font-size","11px")]},
            {"selector":"tr:hover td","props":[("background-color","#EEF2FF")]},
        ]).format(precision=4, na_rep="N/A"))
    display(styler)

if PHASE1_CSV is None or not Path(PHASE1_CSV).exists():
    print(f"  Phase 1 CSV not found. Update _p1_candidates in CONFIG.")
    print(f"  Tried: {[str(p) for p in _p1_candidates]}")
elif PHASE2_CSV is None or not Path(PHASE2_CSV).exists():
    print(f"  Phase 2 CSV not found. Update _p2_candidates in CONFIG.")
    print(f"  Tried: {[str(p) for p in _p2_candidates]}")
else:
    p1_df = pd.read_csv(PHASE1_CSV)
    p2_df = pd.read_csv(PHASE2_CSV)
    p3_df = baseline_df.copy()

    p1_df["dataset"] = "Kaggle CC (P1)"
    p2_df["dataset"] = "IEEE-CIS (P2)"
    p3_df["dataset"] = "Elliptic BTC (P3)"

    compare_metrics = ["pr_auc", "ap", "f1", "recall", "precision", "roc_auc"]

    # ── 13a. 3-way performance comparison ────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle("3-Way Cross-Phase Comparison: Kaggle CC vs IEEE-CIS vs Elliptic Bitcoin",
                 fontsize=14, fontweight="bold")

    PHASE_COLORS = {"Kaggle CC (P1)": "#0072B2",
                    "IEEE-CIS (P2)":  "#009E73",
                    "Elliptic BTC (P3)": "#D55E00"}

    for ax, (metric, ylabel) in zip(axes, [
        ("pr_auc","PR-AUC"), ("ap","Average Precision"), ("f1","F1-Score")
    ]):
        x = np.arange(len(MODEL_NAMES)); w = 0.25
        for j, (df_phase, phase_label) in enumerate([(p1_df,"Kaggle CC (P1)"),
                                                      (p2_df,"IEEE-CIS (P2)"),
                                                      (p3_df,"Elliptic BTC (P3)")]):
            vals = [df_phase.loc[df_phase["model"]==m, metric].values[0]
                    if m in df_phase["model"].values else 0 for m in MODEL_NAMES]
            bars = ax.bar(x + (j-1)*w, vals, width=w,
                          label=phase_label, color=PHASE_COLORS[phase_label],
                          edgecolor="black", alpha=0.85)
            for bar, val in zip(bars, vals):
                if val > 0:
                    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                            f"{val:.3f}", ha="center", fontsize=6.5, fontweight="bold", rotation=90)
        ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, rotation=15, fontsize=9)
        ax.set_ylabel(ylabel); ax.set_title(f"{ylabel} — All 3 Phases")
        ax.set_ylim(0, 1.25)
        if ax == axes[0]: ax.legend(fontsize=9)
    save_fig("cross_phase_01_3way_comparison.png")

    # ── 13b. Delta chart (all phases vs Phase 1 baseline) ────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Performance Delta vs Phase 1 Baseline\n"
                 "Shows how much harder each dataset is relative to Kaggle CC",
                 fontsize=13, fontweight="bold")

    for ax, (metric, ylabel) in zip(axes, [("pr_auc","Delta PR-AUC"),("f1","Delta F1")]):
        x = np.arange(len(MODEL_NAMES)); w = 0.35

        delta_p2 = [p2_df.loc[p2_df["model"]==m, metric].values[0] -
                    p1_df.loc[p1_df["model"]==m, metric].values[0]
                    if m in p2_df["model"].values and m in p1_df["model"].values else 0
                    for m in MODEL_NAMES]
        delta_p3 = [p3_df.loc[p3_df["model"]==m, metric].values[0] -
                    p1_df.loc[p1_df["model"]==m, metric].values[0]
                    if m in p3_df["model"].values and m in p1_df["model"].values else 0
                    for m in MODEL_NAMES]

        b1 = ax.bar(x-w/2, delta_p2, width=w, label="IEEE-CIS (P2) − Kaggle CC (P1)",
                    color="#009E73", edgecolor="black", alpha=0.85)
        b2 = ax.bar(x+w/2, delta_p3, width=w, label="Elliptic BTC (P3) − Kaggle CC (P1)",
                    color="#D55E00", edgecolor="black", alpha=0.85)
        ax.axhline(0, color="black", linewidth=1.2)
        ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, rotation=15)
        ax.set_ylabel(ylabel); ax.set_title(ylabel)
        ax.legend(fontsize=9)
        for bars in [b1, b2]:
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x()+bar.get_width()/2,
                        h+(0.005 if h>=0 else -0.015),
                        f"{h:+.3f}", ha="center", fontsize=8, fontweight="bold")
    save_fig("cross_phase_02_delta_vs_phase1.png")

    # ── 13c. Ranking consistency across all 3 phases ─────────
    rank_data = {}
    for m in MODEL_NAMES:
        r1 = p1_df[p1_df["model"]==m]["pr_auc"].values[0] if m in p1_df["model"].values else 0
        r2 = p2_df[p2_df["model"]==m]["pr_auc"].values[0] if m in p2_df["model"].values else 0
        r3 = p3_df[p3_df["model"]==m]["pr_auc"].values[0] if m in p3_df["model"].values else 0
        rank_data[m] = {"P1": r1, "P2": r2, "P3": r3}

    rank_df_3way = pd.DataFrame(rank_data).T.reset_index()
    rank_df_3way.columns = ["Model","Phase 1","Phase 2","Phase 3"]
    rank_df_3way["Rank P1"] = rank_df_3way["Phase 1"].rank(ascending=False).astype(int)
    rank_df_3way["Rank P2"] = rank_df_3way["Phase 2"].rank(ascending=False).astype(int)
    rank_df_3way["Rank P3"] = rank_df_3way["Phase 3"].rank(ascending=False).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Model Ranking Consistency Across All 3 Datasets",
                 fontsize=14, fontweight="bold")

    x = np.arange(len(MODEL_NAMES)); w = 0.25
    for j, (phase_col, phase_label) in enumerate([
        ("Phase 1","Kaggle CC (P1)"),("Phase 2","IEEE-CIS (P2)"),("Phase 3","Elliptic BTC (P3)")
    ]):
        axes[0].bar(x+(j-1)*w,
                    [rank_df_3way.loc[rank_df_3way["Model"]==m, phase_col].values[0]
                     for m in MODEL_NAMES],
                    width=w, label=phase_label,
                    color=list(PHASE_COLORS.values())[j],
                    edgecolor="black", alpha=0.85)
    axes[0].set_xticks(x); axes[0].set_xticklabels(MODEL_NAMES, rotation=15)
    axes[0].set_ylabel("PR-AUC"); axes[0].set_title("PR-AUC: All Models × All Phases")
    axes[0].legend(fontsize=9); axes[0].set_ylim(0, 1.1)

    for _, row in rank_df_3way.iterrows():
        mname = row["Model"]
        ranks = [row["Rank P1"], row["Rank P2"], row["Rank P3"]]
        axes[1].plot([1,2,3], ranks, color=COLORS.get(mname,"gray"),
                     linewidth=2.5, marker="o", markersize=8, label=mname)
        for j, r in enumerate([1,2,3]):
            axes[1].annotate(f"#{ranks[j-1]}", xy=(r, ranks[j-1]),
                             xytext=(r+0.05, ranks[j-1]),
                             fontsize=9, color=COLORS.get(mname,"gray"))
    axes[1].set_xticks([1,2,3])
    axes[1].set_xticklabels(["Phase 1\n(Kaggle CC)","Phase 2\n(IEEE-CIS)","Phase 3\n(Elliptic)"])
    axes[1].set_yticks([1,2,3,4]); axes[1].set_yticklabels(["#1","#2","#3","#4"])
    axes[1].set_ylabel("Rank (lower = better)")
    axes[1].set_title("Model Ranking Consistency (Bump Chart)\nStable rank = generalizable ordering")
    axes[1].invert_yaxis(); axes[1].legend(fontsize=9)
    save_fig("cross_phase_03_ranking_consistency.png")

    # ── 13d. Fraud rate vs performance scatter ────────────────
    fraud_rates  = [0.1727, 3.5, float(y.mean()*100)]
    phase_labels = ["Kaggle CC\n(P1)","IEEE-CIS\n(P2)","Elliptic\n(P3)"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Fraud Rate vs Model Performance Across 3 Datasets\n"
                 "Shows how fraud prevalence relates to detection difficulty",
                 fontsize=13, fontweight="bold")

    for ax, (metric, ylabel) in zip(axes, [("pr_auc","PR-AUC"),("f1","F1-Score")]):
        for mname in MODEL_NAMES:
            vals = [p1_df.loc[p1_df["model"]==mname, metric].values[0]
                    if mname in p1_df["model"].values else None,
                    p2_df.loc[p2_df["model"]==mname, metric].values[0]
                    if mname in p2_df["model"].values else None,
                    p3_df.loc[p3_df["model"]==mname, metric].values[0]
                    if mname in p3_df["model"].values else None]
            ax.plot(fraud_rates, vals,
                    color=COLORS[mname], linewidth=2, marker=MARKERS[mname],
                    markersize=9, label=mname)
        ax.set_xlabel("Fraud Rate (%)"); ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} vs Fraud Prevalence")
        ax.set_ylim(0, 1); ax.legend(fontsize=9)
        for fr, label in zip(fraud_rates, phase_labels):
            ax.axvline(fr, color="gray", linestyle=":", linewidth=1, alpha=0.5)
            ax.text(fr, 0.02, label, ha="center", fontsize=8, color="gray")
    save_fig("cross_phase_04_fraud_rate_vs_performance.png")

    # ── Styled comparison table ───────────────────────────────
    comp_rows = []
    for mname in MODEL_NAMES:
        r1 = p1_df[p1_df["model"]==mname]["pr_auc"].values[0] if mname in p1_df["model"].values else None
        r2 = p2_df[p2_df["model"]==mname]["pr_auc"].values[0] if mname in p2_df["model"].values else None
        r3 = p3_df[p3_df["model"]==mname]["pr_auc"].values[0] if mname in p3_df["model"].values else None
        comp_rows.append({
            "Model": mname,
            "P1 PR-AUC (Kaggle CC)": r1,
            "P2 PR-AUC (IEEE-CIS)":  r2,
            "P3 PR-AUC (Elliptic)":  r3,
            "P2−P1": (r2-r1) if (r1 is not None and r2 is not None) else None,
            "P3−P1": (r3-r1) if (r1 is not None and r3 is not None) else None,
        })
    comp_df = pd.DataFrame(comp_rows)
    comp_df.to_csv(RES_DIR / "phase3_cross_phase_comparison.csv", index=False)
    styled_table(comp_df,
                 higher_is_better=["P1 PR-AUC (Kaggle CC)","P2 PR-AUC (IEEE-CIS)","P3 PR-AUC (Elliptic)"],
                 title="3-Way Cross-Phase Comparison — PR-AUC (Phase 2 and 3 deltas relative to Phase 1)")

    # Dynamic print_summary
    summary_rows_cross = [
        ("Best model P1", f"{p1_df.sort_values('pr_auc',ascending=False).iloc[0]['model']}  ({p1_df['pr_auc'].max():.4f})"),
        ("Best model P2", f"{p2_df.sort_values('pr_auc',ascending=False).iloc[0]['model']}  ({p2_df['pr_auc'].max():.4f})"),
        ("Best model P3", f"{p3_df.sort_values('pr_auc',ascending=False).iloc[0]['model']}  ({p3_df['pr_auc'].max():.4f})"),
    ]
    for mname in MODEL_NAMES:
        r1 = comp_df[comp_df["Model"]==mname]["P1 PR-AUC (Kaggle CC)"].values[0]
        r3 = comp_df[comp_df["Model"]==mname]["P3 PR-AUC (Elliptic)"].values[0]
        if r1 is not None and r3 is not None:
            summary_rows_cross.append((f"{mname} P3−P1 delta", f"{r3-r1:+.4f}"))
    summary_rows_cross += [
        ("Ranking stable across 3 phases?", "Partially — LR always last; XGB/LGBM compete for #1"),
        ("Key finding", "PR-AUC not monotonic with fraud rate — dataset complexity dominates"),
    ]
    print_summary("3-Way Cross-Phase Comparison", summary_rows_cross)


13) 3-WAY CROSS-PHASE COMPARISON
  [saved] cross_phase_01_3way_comparison.png
  [saved] cross_phase_02_delta_vs_phase1.png
  [saved] cross_phase_03_ranking_consistency.png
  [saved] cross_phase_04_fraud_rate_vs_performance.png


,Model,P1 PR-AUC (Kaggle CC),P2 PR-AUC (IEEE-CIS),P3 PR-AUC (Elliptic),P2−P1,P3−P1
0,Logistic Regression,0.6566,0.2618,0.6721,-0.3947,0.0155
1,Random Forest,0.7519,0.4584,0.9679,-0.2935,0.2160
2,XGBoost,0.8088,0.4763,0.9812,-0.3325,0.1724
3,LightGBM,0.8044,0.5101,0.9848,-0.2943,0.1804



┌────────────────────────────────────────────────────────────┐
│  📊 3-Way Cross-Phase Comparison                           │
├────────────────────────────────────────────────────────────┤
│  Best model P1: XGBoost  (0.8088)                          │
│  Best model P2: LightGBM  (0.5101)                         │
│  Best model P3: LightGBM  (0.9848)                         │
│  Logistic Regression P3−P1 delta: +0.0155                  │
│  Random Forest P3−P1 delta: +0.2160                        │
│  XGBoost P3−P1 delta: +0.1724                              │
│  LightGBM P3−P1 delta: +0.1804                             │
│  Ranking stable across 3 phases?: Partially — LR always last; XGB/LGBM compete for #1│
│  Key finding: PR-AUC not monotonic with fraud rate — dataset complexity dominates│
└────────────────────────────────────────────────────────────┘


## 💾 14 — Metadata Export & Final Summary

In [17]:

# ============================================================
# OPERATIONAL ANALYSIS — Training & Inference Time
# ============================================================
print("\n" + "="*70)
print("OPERATIONAL ANALYSIS — Phase 3 (Elliptic Bitcoin)")
print("="*70)

if not baseline_df.empty:
    _time_cols = [c for c in ["model","train_time_s","infer_time_s"] if c in baseline_df.columns]
    if len(_time_cols) >= 2:
        _time_df = baseline_df[_time_cols].sort_values("train_time_s")
        print("\n  Model | Train (s) | Inference (s)")
        print("  " + "-"*50)
        for _, _row in _time_df.iterrows():
            _inf = f"{_row['infer_time_s']:.4f}" if "infer_time_s" in _time_df.columns else "N/A"
            print(f"  {_row['model']:<25} | {_row['train_time_s']:>9.3f} | {_inf:>14}")

        _fig, _axes = plt.subplots(1, 2, figsize=(14, 5))
        _fig.suptitle("Operational Efficiency — Phase 3 (Elliptic Bitcoin)",
                      fontsize=14, fontweight="bold")
        _models_ord  = _time_df["model"].tolist()
        _colors_ord  = [COLORS[m] for m in _models_ord]
        _axes[0].barh(_models_ord, _time_df["train_time_s"],
                      color=_colors_ord, edgecolor="black", linewidth=0.7)
        _axes[0].set_xlabel("Training Time (seconds)")
        _axes[0].set_title("Training Time by Model")
        _axes[0].invert_yaxis()
        if "infer_time_s" in _time_df.columns:
            _axes[1].barh(_models_ord, _time_df["infer_time_s"],
                          color=_colors_ord, edgecolor="black", linewidth=0.7)
            _axes[1].set_xlabel("Inference Time (seconds)")
            _axes[1].set_title("Inference Time by Model")
            _axes[1].invert_yaxis()
        plt.tight_layout()
        save_fig("operational_01_timing_comparison.png")

        _fastest = _time_df.iloc[0]["model"]
        _slowest = _time_df.iloc[-1]["model"]
        _ratio   = _time_df.iloc[-1]["train_time_s"] / max(_time_df.iloc[0]["train_time_s"], 1e-6)
        print(f"\n  Fastest to train : {_fastest}")
        print(f"  Slowest to train : {_slowest} ({_ratio:.1f}x slower)")

# ── RQ Mapping ────────────────────────────────────────────────
print("\n" + "="*70)
print("RQ MAPPING — Phase 3")
print("="*70)
print("""
RQ1 & RQ2 FINAL SYNTHESIS — All Three Phases Combined:

RQ1 — Gradient boosting vs classical baselines:
  Phase 1 (0.17% fraud, 30 PCA feats)  : see baseline_df Phase 1
  Phase 2 (~3.5% fraud, 224 mixed feats): see baseline_df Phase 2
  Phase 3 (~9.8% fraud, 166 anon feats) : see baseline_df Phase 3
  → Overall: LightGBM shows superior or equivalent PR-AUC across all
    three datasets. Gap is largest at extreme imbalance + rich features.

RQ2 — Data characteristics that impact performance:
  — Imbalance severity: lower fraud rate hurts all models; hurts LR most.
  — Dataset size: gradient boosting advantage clearer at full data size.
  — Feature richness: IEEE-CIS (224 feats) widens the boosting advantage.
  — Domain shift: fraud patterns differ across banking / e-commerce /
    cryptocurrency, but model rankings remain consistent.
  → See Section 13 (3-way cross-phase comparison) for full evidence.""")

# ============================================================
# 14) METADATA EXPORT & FINAL SUMMARY
# ============================================================
meta = {
    "dataset": "Elliptic Bitcoin Transactions",
    "features_csv": str(FEATURES_CSV),
    "n_labeled": int(len(df_labeled)),
    "fraud_rate": float(y.mean()),
    "n_features": len(FEATURE_COLS),
    "time_steps": int(df_labeled["time_step"].nunique()),
    "n_graph_edges": int(len(df_edges)),
    "splits": {"train": 1-TEST_SIZE-VAL_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
    "models_tested": MODEL_NAMES,
    "imbalance_strategy": "class_weight=balanced (no SMOTE/resampling)",
    "sensitivity": {
        "imbalance_ratios": IMB_RATIOS,
        "size_fractions": SIZE_FRACS,
        "repeats": N_REPEATS,
        "bootstrap_samples": N_BOOTSTRAP,
    },
    "best_model_by_pr_auc": str(baseline_df.iloc[0]["model"]),
    "best_pr_auc": float(baseline_df.iloc[0]["pr_auc"]),
    "calibration": {m: {"brier": v["brier"], "ece": v["ece"]}
                    for m, v in calib_stats.items()},
}
with open(RES_DIR / "phase3_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# Final results table — use numeric values only for styled_table
summary_rows = []
for _, row in baseline_df.iterrows():
    summary_rows.append({
        "Model":      row["model"],
        "PR-AUC":     round(row["pr_auc"], 4),
        "CI low":     round(row["pr_auc_ci_lo"], 4),
        "CI high":    round(row["pr_auc_ci_hi"], 4),
        "AP":         round(row["ap"], 4),
        "F1":         round(row["f1"], 4),
        "Recall":     round(row["recall"], 4),
        "Precision":  round(row["precision"], 4),
        "Train(s)":   round(row["train_time_s"], 2),
        "Brier":      round(calib_stats[row["model"]]["brier"], 4) if row["model"] in calib_stats else None,
        "ECE":        round(calib_stats[row["model"]]["ece"], 4)   if row["model"] in calib_stats else None,
    })
final_df = pd.DataFrame(summary_rows)
styled_table(final_df,
             higher_is_better=["PR-AUC","AP","F1","Recall","Precision"],
             title="Phase 3 — Complete Results Table (Elliptic Bitcoin, 95% CI in brackets)")

print("\n" + "="*70)
print("PHASE 3 COMPLETE ✅")
print("="*70)
figs = sorted(FIG_DIR.glob("*.png"))
print(f"\n{len(figs)} figures saved to: {FIG_DIR}")
for p in figs: print(f"  {p.name}")
print(f"\nResults saved to: {RES_DIR}")

print_summary("Phase 3 Complete — Elliptic Bitcoin", [
    ("Dataset", "Elliptic Bitcoin Transactions"),
    ("Labeled transactions", f"{len(df_labeled):,}"),
    ("Best model overall", f"{baseline_df.iloc[0]['model']} (PR-AUC={baseline_df.iloc[0]['pr_auc']:.4f})"),
    ("Best calibration", f"{calib_df.iloc[0]['Model']} (ECE={calib_df.iloc[0]['ECE']:.4f})" if "calib_df" in dir() else "N/A"),
    ("Most stable", f"{rob_summary.loc[rob_summary['pr_auc_std'].idxmin(),'model']} (std={rob_summary['pr_auc_std'].min():.4f})"),
    ("Temporal split best", f"{temp_df.sort_values('pr_auc',ascending=False).iloc[0]['model']}" if "temp_df" in dir() else "N/A"),
    ("Outputs saved to", str(OUT_DIR)),
])


OPERATIONAL ANALYSIS — Phase 3 (Elliptic Bitcoin)

  Model | Train (s) | Inference (s)
  --------------------------------------------------
  Random Forest             |     2.189 |         0.0740
  LightGBM                  |     2.250 |         0.0285
  Logistic Regression       |     5.806 |         0.0233
  XGBoost                   |     5.962 |         0.0403
  [saved] operational_01_timing_comparison.png

  Fastest to train : Random Forest
  Slowest to train : XGBoost (2.7x slower)

RQ MAPPING — Phase 3

RQ1 & RQ2 FINAL SYNTHESIS — All Three Phases Combined:

RQ1 — Gradient boosting vs classical baselines:
  Phase 1 (0.17% fraud, 30 PCA feats)  : see baseline_df Phase 1
  Phase 2 (~3.5% fraud, 224 mixed feats): see baseline_df Phase 2
  Phase 3 (~9.8% fraud, 166 anon feats) : see baseline_df Phase 3
  → Overall: LightGBM shows superior or equivalent PR-AUC across all
    three datasets. Gap is largest at extreme imbalance + rich features.

RQ2 — Data characteristics that impact

,Model,PR-AUC,CI low,CI high,AP,F1,Recall,Precision,Train(s),Brier,ECE
0,LightGBM,0.9848,0.9788,0.9895,0.9848,0.9507,0.9326,0.9695,2.2500,0.0094,0.0143
1,XGBoost,0.9812,0.9745,0.9869,0.9812,0.9372,0.9076,0.9687,5.9600,0.0138,0.0254
2,Random Forest,0.9679,0.9588,0.9761,0.9679,0.9251,0.8783,0.9772,2.1900,0.0209,0.0593
3,Logistic Regression,0.6721,0.6333,0.7150,0.6729,0.7160,0.6950,0.7383,5.8100,0.0975,0.1443



PHASE 3 COMPLETE ✅

33 figures saved to: c:\Users\Nouha Laataoui\OneDrive - International School Of Elite - ISE\Bureau\My GP\Phase 1\output phase 3\figures
  baseline_01_pr_curves.png
  baseline_02_roc_curves.png
  baseline_03_confusion_matrices.png
  baseline_04_performance_bars_CI.png
  baseline_05_timing.png
  calibration_01_reliability_diagram.png
  calibration_02_scores.png
  cross_phase_01_3way_comparison.png
  cross_phase_02_delta_vs_phase1.png
  cross_phase_03_ranking_consistency.png
  cross_phase_04_fraud_rate_vs_performance.png
  eda_01_class_distribution.png
  eda_02_temporal_analysis.png
  eda_03_top6_feature_violins.png
  eda_04_graph_structure.png
  fi_01_top15_bars.png
  fi_02_local_vs_aggregated.png
  fi_03_consensus_heatmap.png
  fi_04_permutation_importance.png
  operational_01_timing_comparison.png
  robustness_01_mean_pr_curves.png
  robustness_02_stability_dotplot.png
  sensitivity_01_imbalance_line.png
  sensitivity_02_imbalance_heatmap.png
  sensitivity_03_size_